## Evaluating Multimodal LLMs for Classical Sheet Music Interpretation
### Katherine (ZhaoYu) Tu — Master Thesis Pipeline
**University of Amsterdam, 2026**

This notebook runs the full benchmark pipeline across three SRQs:
- **SRQ1**: Key + time signature identification (zero-shot, PDF input)
- **SRQ2**: Key + time signature identification (CoT, PDF input)
- **SRQ3**: Key + time signature identification across input formats (PDF vs PNG)

Models evaluated: `claude-sonnet-4-6`, `gemini-2.5-flash`, `gpt-5.4`

In [ ]:
import sys
print(sys.executable)

In [ ]:
import os
import io
import json
import time
import base64
import requests
import anthropic
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from sklearn.metrics import precision_recall_fscore_support

#import google.generativeai as genai
print("All packages imported successfully")

## Setup

Install dependencies once from the project root before running this notebook:

```bash
pip install -r requirements.txt
```

You'll also need a `.env` file at the project root with your API keys and corpus location. See `.env.example` and `README.md` for details.

### Configure paths, retry settings, and model names.

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────
BASE_DIR = Path(os.getenv("CORPUS_DIR", "./data/beethoven_piano_sonatas"))
HARMONIES_DIR    = BASE_DIR / "harmonies"
MEASURES_DIR     = BASE_DIR / "measures"
NOTES_DIR        = BASE_DIR / "notes"
PDF_DIR          = BASE_DIR / "exports_pdf"
PNG_DIR          = BASE_DIR / "exports_png"
RESULTS_DIR      = BASE_DIR / "results"
GROUND_TRUTH_FILE = BASE_DIR / "ground_truth.json"

# Few-shot examples directory (Mozart sonatas PDFs — place manually here)
FEWSHOT_DIR      = BASE_DIR / "fewshot_examples"

# Create results folder if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── MODEL NAMES ──────────────────────────────────────────────────────────────
CLAUDE_MODEL = "claude-sonnet-4-6"
GEMINI_MODEL = "gemini-2.5-flash"
GPT_MODEL    = "gpt-5.4"

# ── RETRY SETTINGS ───────────────────────────────────────────────────────────
MAX_RETRIES  = 3
RETRY_DELAY  = 3  # seconds between retries

print("Config loaded.")
print(f"PDF dir exists: {PDF_DIR.exists()}")
print(f"PNG dir exists: {PNG_DIR.exists()}")

### Load API keys from .env file and initialise clients.

Create a `.env` file in your project root with:
```
ANTHROPIC_API_KEY=your_key_here
GEMINI_API_KEY=your_key_here
OPENAI_API_KEY=your_key_here
```

In [ ]:
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

# Initialise clients
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client    = OpenAI(api_key=OPENAI_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Sanity check — keys are loaded (not printed for security)
assert ANTHROPIC_API_KEY, "ANTHROPIC_API_KEY not found in .env"
assert GEMINI_API_KEY,    "GEMINI_API_KEY not found in .env"
assert OPENAI_API_KEY,    "OPENAI_API_KEY not found in .env"
print("All API keys loaded successfully.")

---
## Part 1 — Ground Truth Extraction
### Extract ground truth from TSV files for all movements.

In [ ]:
def get_movement_ids():
    """Return sorted list of all movement IDs e.g. ['01-1', '01-2', ...]"""
    files = list(MEASURES_DIR.glob("*.measures.tsv"))
    ids = sorted([f.name.replace(".measures.tsv", "") for f in files])
    return ids

def extract_time_signature(movement_id):
    """Get time signature from first measure of measures TSV."""
    path = MEASURES_DIR / f"{movement_id}.measures.tsv"
    try:
        df = pd.read_csv(path, sep="\t", low_memory=False)
        timesig = df["timesig"].dropna().iloc[0]
        return str(timesig)
    except Exception as e:
        print(f"  [WARN] time sig failed for {movement_id}: {e}")
        return None

def extract_key_signature(movement_id):
    """Get global key from harmonies TSV. Returns None if unannotated."""
    path = HARMONIES_DIR / f"{movement_id}.harmonies.tsv"
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path, sep="\t", low_memory=False)
        if "globalkey" not in df.columns:
            return None
        key = df["globalkey"].dropna()
        if key.empty:
            return None
        return str(key.iloc[0])
    except Exception as e:
        print(f"  [WARN] key sig failed for {movement_id}: {e}")
        return None

def extract_highest_pitch_bar1(movement_id):
    """Get the highest sounding pitch (by MIDI number) in measure 1."""
    path = NOTES_DIR / f"{movement_id}.notes.tsv"
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path, sep="\t", low_memory=False)
        # mc = measure count column in DCML corpus
        bar1 = df[df["mc"] == 1]
        if bar1.empty:
            return None
        highest_row = bar1.loc[bar1["midi"].idxmax()]
        # tpc = tonal pitch class (e.g. 'F', 'C#')
        # midi = MIDI number used to find highest
        return {
            "tpc":  str(highest_row["tpc"]),
            "midi": int(highest_row["midi"])
        }
    except Exception as e:
        print(f"  [WARN] pitch failed for {movement_id}: {e}")
        return None

def build_ground_truth():
    """Build and save ground truth JSON for all movements."""
    movement_ids = get_movement_ids()
    print(f"Found {len(movement_ids)} movements.")

    results = []
    for mid in movement_ids:
        entry = {
            "movement_id":        mid,
            "pdf_file":           f"{mid}.pdf",
            "png_file":           f"{mid}.png",
            "time_signature":     extract_time_signature(mid),
            "key_signature":      extract_key_signature(mid),
            "highest_pitch_bar1": extract_highest_pitch_bar1(mid),
        }
        results.append(entry)

    with open(GROUND_TRUTH_FILE, "w") as f:
        json.dump(results, f, indent=2)

    with_key  = sum(1 for r in results if r["key_signature"])
    with_time = sum(1 for r in results if r["time_signature"])
    with_pitch = sum(1 for r in results if r["highest_pitch_bar1"])

    print(f"\nGround truth saved to: {GROUND_TRUTH_FILE}")
    print(f"  Total movements:         {len(results)}")
    print(f"  With key signature:      {with_key}")
    print(f"  With time signature:     {with_time}")
    print(f"  With pitch (bar 1):      {with_pitch}")
    return results

ground_truth = build_ground_truth()

### Cell 5 — Quick sanity check on ground truth.

In [ ]:
# Print first 5 entries to verify
for entry in ground_truth[:5]:
    print(json.dumps(entry, indent=2))
    print("---")

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PIECE_TO_FIX   = "10-2"
FIELD_TO_FIX   = "time_signature"   # question_type value in your JSONL
OLD_VALUE      = "4/4"
NEW_VALUE      = "2/2"

FILES_TO_PATCH = [
    SRQ1_RESULTS_FILE,
    SRQ2_RESULTS_FILE,
    SRQ3_RESULTS_FILE,
]

# ── PATCH ─────────────────────────────────────────────────────────────────────
for filepath in FILES_TO_PATCH:
    path = Path(filepath)
    if not path.exists():
        print(f"[SKIP] Not found: {path.name}")
        continue

    rows        = [json.loads(line) for line in path.open() if line.strip()]
    n_patched   = 0
    patched_rows = []

    for row in rows:
        if (row.get("piece_id")      == PIECE_TO_FIX and
            row.get("question_type") == FIELD_TO_FIX and
            row.get("ground_truth")  == OLD_VALUE):

            row["ground_truth"] = NEW_VALUE

            # Recompute correct flag with the new ground truth
            # Uses your existing normalise_for_sklearn / correctness logic
            parsed = row.get("parsed_answer", "")
            row["correct"] = (str(parsed).strip().lower() ==
                              NEW_VALUE.strip().lower())
            n_patched += 1

        patched_rows.append(row)

    # Write back
    with path.open("w") as f:
        for row in patched_rows:
            f.write(json.dumps(row) + "\n")

    print(f"{path.name}: patched {n_patched} rows")

---
## Part 2 — Prompt Builders
### Cell 6 — Define all prompt variants for each task.

In [ ]:
# ── SRQ1 PROMPTS (zero-shot only) ─────────────────────────────────────────────

def prompt_srq1_key():
    return (
        "Below is a piano score in PDF format. "
        "What is the key signature of this piece? "
        "Respond with ONLY the key signature, no explanation. "
        "Example format: 'F minor' or 'A major'."
    )

def prompt_srq1_time():
    return (
        "Below is a piano score in PDF format. "
        "What is the time signature of this piece? "
        "Respond with ONLY the time signature, no explanation. "
        "Example format: '4/4' or '3/4'."
    )

# ── SRQ2: CoT FAITHFULNESS PROMPTS ───────────────────────────────────────
# These prompts force explicit step-by-step reasoning.
# The model must end with 'Final answer: X' so we can parse it reliably.
# The reasoning steps before the final answer are what we analyse for faithfulness.

def prompt_srq2_cot_key():
    return (
        "Below is a piano score. "
        "I want you to identify the key signature by thinking step by step.\n"
        "(1) Look at the beginning of the staff. "
        "Count and name every sharp or flat symbol shown in the key signature. "
        "List them explicitly — e.g. 'I see 2 flats: Bb and Eb'.\n"
        "(2) Based on the accidentals you identified, determine which "
        "major or minor key this corresponds to. Show your reasoning.\n"
        "(3) State your final answer.\n"
        "End your response with exactly: 'Final answer: [key name]' "
        "(e.g. 'Final answer: F minor' or 'Final answer: A major')."
    )

def prompt_srq2_cot_time():
    return (
        "Below is a piano score. "
        "I want you to identify the time signature by thinking step by step.\n"
        "(1) Look at the beginning of the staff, after the clef and key signature. "
        "Describe the time signature symbol you see — the two numbers stacked vertically.\n"
        "(2) State what the top number (beats per bar) and "
        "bottom number (note value of one beat) represent.\n"
        "(3) State your final answer.\n"
        "End your response with exactly: 'Final answer: [time signature]' "
        "(e.g. 'Final answer: 4/4' or 'Final answer: 3/4')."
    )

print("CoT faithfulness prompt builders defined.")

# ── SRQ3 PROMPTS (zero-shot + role, same as SRQ1 but for both formats) ────────
# SRQ3 uses the same prompts as SRQ1 but with role priming added

def prompt_srq3_key():
    return (
        "You are an expert music theory teacher. "
        "Below is a piano score. "
        "What is the key signature of this piece? "
        "Respond with ONLY the key signature, no explanation. "
        "Example format: 'F minor' or 'A major'."
    )

def prompt_srq3_time():
    return (
        "You are an expert music theory teacher. "
        "Below is a piano score. "
        "What is the time signature of this piece? "
        "Respond with ONLY the time signature, no explanation. "
        "Example format: '4/4' or '3/4'."
    )

print("Prompt builders defined.")

---
## Part 4 — Model Query Functions
### Cell 8 — Retry wrapper.

In [ ]:
def retry_call(func, *args):
    """Retry a function up to MAX_RETRIES times on ERROR response."""
    for attempt in range(MAX_RETRIES):
        result = func(*args)
        if result != "ERROR":
            return result
        print(f"  Retry {attempt + 1}/{MAX_RETRIES}...")
        time.sleep(RETRY_DELAY)
    return "ERROR"

print("Retry wrapper defined.")

### Cell 9 — Claude (Anthropic) query function.

In [ ]:
def encode_pdf(pdf_path):
    """Helper: encode a PDF file to base64 string."""
    with open(pdf_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def query_claude(pdf_path, question_text, fewshot_examples=None):
    """
    Query Claude with a PDF score and a question.
    Optionally pass fewshot_examples (list of {pdf_b64, answer}) for few-shot.
    """
    try:
        content = []

        # Add few-shot examples if provided
        if fewshot_examples:
            for ex in fewshot_examples:
                content.append({
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": ex["pdf_b64"]
                    }
                })
                content.append({
                    "type": "text",
                    "text": f"Example answer: {ex['answer']}"
                })

        # Add the actual test PDF
        content.append({
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": encode_pdf(pdf_path)
            }
        })

        # Add the question
        content.append({"type": "text", "text": question_text})

        response = anthropic_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=512,
            temperature=0,
            timeout=30,         
            messages=[{"role": "user", "content": content}]
        )
        return response.content[0].text.strip()

    except Exception as e:
        print(f"  Claude error: {e}")
        return "ERROR"

print("Claude query function defined.")

### Cell 10 — Gemini (Google) query function.

In [ ]:
def query_gemini(pdf_path, question_text, fewshot_examples=None):
    try:
        parts = []
        if fewshot_examples:
            for ex in fewshot_examples:
                parts.append(base64.b64decode(ex["pdf_b64"]))
                parts.append(f"Example answer: {ex['answer']}")
        
        with open(pdf_path, "rb") as f:
            pdf_bytes = f.read()

        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                {
                    "parts": [
                        {"inline_data": {"mime_type": "application/pdf", "data": base64.b64encode(pdf_bytes).decode()}},
                        {"text": question_text}
                    ]
                }
            ],
            config={
                "temperature": 0,
                "http_options": {"timeout": 30000}
            }
        )
        return response.text.strip()

    except Exception as e:
        print(f"  Gemini error: {e}")
        return "ERROR"

print("Gemini query function defined.")

### Cell 11 — GPT-5.4 (OpenAI) query function.
GPT-5.4 uses the Files API: PDFs are uploaded first, then referenced by file ID.
A cache avoids re-uploading the same file multiple times.

In [ ]:
# Cache: pdf_path (str) → file_id (str)
# Avoids re-uploading the same PDF on every call
gpt_file_cache = {}

def get_gpt_file_id(pdf_path):
    """Upload PDF to OpenAI Files API and return file_id. Cached per path."""
    path_str = str(pdf_path)
    if path_str not in gpt_file_cache:
        with open(pdf_path, "rb") as f:
            upload = openai_client.files.create(file=f, purpose="user_data")
        gpt_file_cache[path_str] = upload.id
        print(f"  Uploaded to OpenAI: {pdf_path.name} → {upload.id}")
    return gpt_file_cache[path_str]

def query_gpt(pdf_path, question_text, fewshot_examples=None):
    """
    Query GPT-5.4 with a PDF score and a question.
    Optionally pass fewshot_examples (list of {pdf_b64, answer}) for few-shot.
    Note: few-shot PDFs for GPT are uploaded via temp base64 inline approach.
    """
    try:
        content = []

        # Add few-shot examples if provided
        if fewshot_examples:
            for ex in fewshot_examples:
                content.append({
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:application/pdf;base64,{ex['pdf_b64']}"
                    }
                })
                content.append({
                    "type": "text",
                    "text": f"Example answer: {ex['answer']}"
                })

        # Add the actual test PDF via file ID
        file_id = get_gpt_file_id(pdf_path)
        content.append({
            "type": "file",
            "file": {"file_id": file_id}
        })

        # Add the question
        content.append({"type": "text", "text": question_text})

        response = openai_client.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": content}],
            temperature=0,
            timeout=30
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"  GPT error: {e}")
        return "ERROR"

print("GPT-5.4 query function defined.")

### Cell 12 — PNG query wrapper (for SRQ3).
SRQ3 compares PDF vs PNG input — same models, same prompts, different file format.

In [ ]:
# ── HELPER: collect all PNG pages for a movement ─────────────────────────────

def get_png_pages(movement_id):
    """
    Return a sorted list of PNG page paths for a given movement_id.
    Naming convention: {movement_id}-{page_number}.png
    e.g. movement "01-1" → [01-1-1.png, 01-1-2.png, ..., 01-1-N.png]
    """
    pages = sorted(
        PNG_DIR.glob(f"{movement_id}-*.png"),
        key=lambda p: int(p.stem.split("-")[-1])  # sort by page number
    )
    return pages

# Quick sanity check
test_pages = get_png_pages("01-1")
print(f"Pages found for 01-1: {[p.name for p in test_pages]}")
assert len(test_pages) > 0, "No PNG pages found for 01-1 — check PNG_DIR and naming convention"


# ── PNG QUERY FUNCTIONS ───────────────────────────────────────────────────────
# All three accept a LIST of png_paths (one per page), sent together in one call.

# Claude's limit for many-image requests
CLAUDE_MAX_DIM = 1999  # stay just under 2000px to be safe

def resize_image_for_claude(img_bytes):
    """
    Resize image in memory so neither dimension exceeds CLAUDE_MAX_DIM.
    Only resizes if necessary. Returns base64-encoded PNG bytes.
    Preserves aspect ratio.
    """
    img = Image.open(io.BytesIO(img_bytes))
    w, h = img.size

    if w > CLAUDE_MAX_DIM or h > CLAUDE_MAX_DIM:
        scale = min(CLAUDE_MAX_DIM / w, CLAUDE_MAX_DIM / h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        img = img.resize((new_w, new_h), Image.LANCZOS)

    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")


def query_claude_png(png_paths, question_text):
    """
    Query Claude with multiple PNG pages as a single message.
    Images are resized in memory to stay within Claude's 2000px dimension limit.
    png_paths: list of Path objects, one per page.
    """
    try:
        content = []

        for png_path in png_paths:
            with open(png_path, "rb") as f:
                img_bytes = f.read()

            img_b64 = resize_image_for_claude(img_bytes)

            content.append({
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": img_b64
                }
            })

        content.append({"type": "text", "text": question_text})

        response = anthropic_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=512,
            timeout=60,
            messages=[{"role": "user", "content": content}]
        )
        return response.content[0].text.strip()

    except Exception as e:
        print(f"  Claude PNG error: {e}")
        return "ERROR"

print("query_claude_png updated with resize support.")

def query_gemini_png(png_paths, question_text):
    """
    Query Gemini with multiple PNG pages as a single message.
    Uses new google-genai SDK (gemini_client), NOT the old GenerativeModel API.
    png_paths: list of Path objects, one per page.
    """
    try:
        parts = []

        for png_path in png_paths:
            with open(png_path, "rb") as f:
                img_bytes = f.read()
            parts.append({
                "inline_data": {
                    "mime_type": "image/png",
                    "data": base64.b64encode(img_bytes).decode()
                }
            })

        parts.append({"text": question_text})

        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[{"parts": parts}],
            config={"http_options": {"timeout": 60000}}
        )
        return response.text.strip()

    except Exception as e:
        print(f"  Gemini PNG error: {e}")
        return "ERROR"


def query_gpt_png(png_paths, question_text):
    """
    Query GPT with multiple PNG pages as a single message.
    png_paths: list of Path objects, one per page.
    """
    try:
        content = []

        for png_path in png_paths:
            with open(png_path, "rb") as f:
                img_b64 = base64.standard_b64encode(f.read()).decode("utf-8")
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{img_b64}"}
            })

        content.append({"type": "text", "text": question_text})

        response = openai_client.chat.completions.create(
            model=GPT_MODEL,
            messages=[{"role": "user", "content": content}],
            temperature=0,
            timeout=60
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"  GPT PNG error: {e}")
        return "ERROR"


print("PNG query functions defined (multi-page support).")

---
## Part 5 — Answer Parser
### Cell 13 — Normalize free-text model responses into clean answer tokens.

In [ ]:
import re

# ── PARSERS ───────────────────────────────────────────────────────────────────

def parse_key_signature(response):
    """
    Extract key signature from model response.
    Handles: 'F minor', 'A major', 'f', 'F#m', 'The key is G major.'
    Returns normalized string e.g. 'f major', 'f minor'
    """
    if not response or response == "ERROR":
        return None
    text = response.lower().strip()

    # Match patterns like 'f minor', 'a major', 'c# minor', 'e-flat major'
    pattern = r'([a-g][#b♯♭]?[-\s]?(?:flat|sharp)?)[\s-]*(major|minor|maj|min|m\b)'
    match = re.search(pattern, text)
    if match:
        note = match.group(1).strip().replace("-", "").replace(" ", "")
        quality = "major" if match.group(2) in ["major", "maj"] else "minor"
        return f"{note} {quality}"

    # If model just returned a single letter (e.g. 'f' from DCML format)
    single = re.match(r'^[a-g][#b]?$', text.strip())
    if single:
        return text.strip()

    # Fallback: return cleaned text as-is
    return text.strip()

def parse_time_signature(response):
    """
    Extract time signature from model response.
    Handles: '4/4', '3/4', 'The time signature is 4/4.', 'common time'

    Picks the FIRST fraction-shaped pattern in the response, since some models
    (notably Claude in CoT mode) follow their answer with explanatory prose
    like "The time signature is 3/4. This is consistent with..." — and we
    want '3/4', not whatever sentence comes after.
    """
    if not response or response == "ERROR":
        return None
    text = response.strip()

    # Match the first standard fraction format.
    # Restricting denominators to powers of 2 (1,2,4,8,16,32) avoids
    # accidentally matching things like "3 quarter-note beats per bar"
    # if a stray pattern slipped through.
    match = re.search(r"\b(\d{1,2})\s*/\s*(\d{1,2})\b", text)
    if match:
        return f"{match.group(1)}/{match.group(2)}"

    # Handle common/cut time written out as words
    if "common" in text.lower():
        return "4/4"
    if "cut" in text.lower() or "alla breve" in text.lower():
        return "2/2"

    return text.strip()

def parse_pitch(response):
    """
    Extract pitch from model response.
    Handles: 'G4', 'D5', 'The highest note is B4.', 'Final answer: C5'
    """
    if not response or response == "ERROR":
        return None
    text = response.strip()

    # Look for 'Final answer: X' pattern (CoT responses)
    final = re.search(r'final answer[:\s]+([A-Ga-g][#b]?\d)', text, re.IGNORECASE)
    if final:
        return final.group(1)

    # Standard pitch pattern: letter + optional accidental + octave number
    match = re.search(r'\b([A-Ga-g][#b♯♭]?\d)\b', text)
    if match:
        return match.group(1)

    return text.strip()

# ── CoT RESPONSE PARSER ───────────────────────────────────────────────────────
# Extracts the final answer from a CoT response.
# Looks for 'Final answer: X' pattern (case-insensitive).
# Falls back to standard parsers if pattern not found.

def parse_cot_response(raw_response, question_type):
    """
    Extract final answer from a CoT response.

    Tries 'Final answer: X' pattern first.
    Falls back to the standard parser for the task if not found.

    Strips markdown formatting (**, *, `, _) before regex matching,
    since some models (notably Claude) wrap their final answer in bold,
    which breaks downstream parsing.

    Uses the LAST occurrence of 'Final answer:' in case the model writes
    a section header like '(3) Final answer:' before the actual answer line.

    Returns (final_answer_str, full_reasoning_str)
    so the reasoning can be saved separately for manual coding.
    """
    if not raw_response or raw_response == "ERROR":
        return None, raw_response

    # Strip markdown formatting characters before parsing.
    cleaned = re.sub(r"[*`_]+", "", raw_response)

    # Find ALL 'Final answer: X' matches and take the last non-empty one.
    # Some models (notably Claude) write '(3) Final answer:' as a section
    # header on its own line, then put the actual answer further down.
    matches = re.findall(
        r"final answer\s*:\s*(.+?)(?:\n|$)",
        cleaned,
        re.IGNORECASE
    )
    # Filter out empty captures (e.g. header lines with nothing on the same line)
    non_empty = [m.strip().rstrip(".") for m in matches if m.strip()]

    if non_empty:
        final_answer = non_empty[-1]   # use the LAST one
    else:
        final_answer = None

    # Run through standard parsers for normalisation.
    if question_type == "key_signature":
        parsed = parse_key_signature(final_answer) if final_answer \
                 else parse_key_signature(cleaned)
    else:
        parsed = parse_time_signature(final_answer) if final_answer \
                 else parse_time_signature(cleaned)

    # Reasoning = everything before the LAST 'Final answer:' in cleaned text
    splits = re.split(r"final answer\s*:", cleaned, flags=re.IGNORECASE)
    reasoning = "final answer:".join(splits[:-1]).strip() if len(splits) > 1 else cleaned

    return parsed, reasoning

def dcml_key_to_standard(dcml_key):
    """
    Convert DCML key string to (note, quality) tuple.
    'F'  → ('f', 'major')
    'f'  → ('f', 'minor')
    'Bb' → ('bb', 'major')
    'bb' → ('bb', 'minor')
    """
    if not dcml_key:
        return None, None
    quality = "major" if dcml_key[0].isupper() else "minor"
    note = dcml_key.lower()
    return note, quality

def normalise_key_for_comparison(raw_key):
    """
    Normalise a model response to (note, quality) tuple.
    Handles: 'F major', 'f minor', 'B-flat major', 'Bb minor',
             'F#', 'f#', 'B♭ major', 'bflat minor'
    """
    if not raw_key:
        return None, None

    raw = raw_key.strip()

    # Short DCML-style key with no spaces (e.g. 'F', 'f', 'Bb', 'f#')
    if len(raw) <= 3 and " " not in raw:
        return dcml_key_to_standard(raw)

    # Parse longer model responses like 'F major', 'b-flat minor'
    raw_lower = raw.lower()
    if "major" in raw_lower or "maj" in raw_lower:
        quality = "major"
    elif "minor" in raw_lower or "min" in raw_lower:
        quality = "minor"
    else:
        quality = "major"  # default if quality not stated

    # Extract and clean note name
    note = raw_lower
    for word in ["major", "minor", "maj", "min"]:
        note = note.replace(word, "")
    note = note.strip().rstrip("-").strip()

    # Normalise written flat/sharp forms
    note = note.replace("flat", "b").replace("sharp", "#")
    note = note.replace("♭", "b").replace("♯", "#")
    note = note.replace("-", "").replace(" ", "")

    return note, quality

def keys_match(ground_truth_val, parsed_answer):
    """
    Compare DCML ground truth key with model parsed answer.
    Returns True only if both note AND quality match.
    """
    if not ground_truth_val or not parsed_answer:
        return False
    gt_note,   gt_quality   = dcml_key_to_standard(str(ground_truth_val))
    pred_note, pred_quality = normalise_key_for_comparison(parsed_answer)
    return gt_note == pred_note and gt_quality == pred_quality

# ── TESTS ─────────────────────────────────────────────────────────────────────

# Parser tests
assert parse_time_signature("The time signature is 4/4.") == "4/4"
assert parse_time_signature("3/4") == "3/4"
assert parse_pitch("Final answer: G4") == "G4"
assert parse_pitch("The highest note is B4.") == "B4"

# Key matching tests
assert keys_match("F",  "F major")       == True
assert keys_match("f",  "f minor")       == True
assert keys_match("F",  "f minor")       == False   # major vs minor
assert keys_match("f",  "F major")       == False   # minor vs major
assert keys_match("Bb", "B-flat major")  == True
assert keys_match("bb", "B-flat minor")  == True
assert keys_match("Bb", "B-flat minor")  == False   # major vs minor
assert keys_match("C",  "c major")       == True
assert keys_match("c",  "C major")       == False
assert keys_match("F#", "F# major")      == True
assert keys_match("f#", "f# minor")      == True

print("All parser and key matching tests passed.")

---
## Part 6 — Results Logger
### Cell 14 — Save results to JSONL incrementally (crash-safe).

In [ ]:
def save_result(filepath, entry):
    """
    Append a single result entry to a JSONL file.
    Saves after every response so a crash doesn't lose data.
    Schema: piece_id, model, prompt_variant, input_type,
            question_type, ground_truth, model_response,
            parsed_answer, correct
    """
    with open(filepath, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

def load_completed(filepath):
    """
    Load already-completed (movement_id, model, prompt_variant, question_type)
    combos from an existing JSONL file. Used to skip already-done items
    if the run was interrupted and you need to resume.
    """
    completed = set()
    if not Path(filepath).exists():
        return completed
    with open(filepath, "r") as f:
        for line in f:
            entry = json.loads(line.strip())
            key = (
                entry["piece_id"],
                entry["model"],
                entry["prompt_variant"],
                entry["question_type"]
            )
            completed.add(key)
    return completed

print("Results logger defined.")

---
## Part 7 — SRQ1: Key + Time Signature (Zero-shot, PDF)
### Cell 15 — Run SRQ1 across all annotated movements and all three models.

In [ ]:
SRQ1_RESULTS_FILE = RESULTS_DIR / "srq1_results_fixed.jsonl"

def run_srq1():
    # Load ground truth
    with open(GROUND_TRUTH_FILE) as f:
        gt_data = json.load(f)

    # Only use movements with key annotation for key task
    # All movements usable for time signature
    completed = load_completed(SRQ1_RESULTS_FILE)

    models = {
        "claude-sonnet-4-6": query_claude,
        "gemini-2.5-flash":  query_gemini,
        "gpt-5.4":           query_gpt
    }

    tasks = [
        ("key_signature",  prompt_srq1_key(),  parse_key_signature,  "key_signature"),
        ("time_signature", prompt_srq1_time(), parse_time_signature, "time_signature")
    ]

    total = len(gt_data) * len(models) * len(tasks)
    done  = len(completed)

    print(f"Total expected entries: {total}")
    print(f"Already completed:      {done}")
    print(f"Remaining:              {total - done}")

    for gt in gt_data:
        mid      = gt["movement_id"]
        pdf_path = PDF_DIR / gt["pdf_file"]

        if not pdf_path.exists():
            print(f"  [SKIP] PDF not found: {pdf_path.name}")
            continue

        for model_name, query_fn in models.items():
            for gt_field, prompt_text, parser_fn, q_type in tasks:

                # Skip if ground truth not available for this task
                if gt[gt_field] is None:
                    continue

                # Skip if already completed (resume support)
                key = (mid, model_name, "zero_shot", q_type)
                if key in completed:
                    done += 1
                    continue

                # Query model
                raw_response = retry_call(query_fn, pdf_path, prompt_text)
                parsed       = parser_fn(raw_response)
                ground_truth_val = gt[gt_field]

                # Check correctness
                if q_type == "key_signature":
                    correct = keys_match(ground_truth_val, parsed)
                else:
                    correct = (
                        parsed is not None and
                        parsed.strip() == str(ground_truth_val).strip()
                    )

                entry = {
                    "piece_id":       mid,
                    "model":          model_name,
                    "prompt_variant": "zero_shot",
                    "input_type":     "pdf",
                    "question_type":  q_type,
                    "ground_truth":   str(ground_truth_val),
                    "model_response": raw_response,
                    "parsed_answer":  parsed,
                    "correct":        correct
                }

                save_result(SRQ1_RESULTS_FILE, entry)
                done += 1
                print(f"  [{done}/{total}] {mid} | {model_name} | {q_type} | correct={correct}")

    print(f"\nSRQ1 done. Results saved to {SRQ1_RESULTS_FILE}")

# ── SAFETY GUARD ──────────────────────────────────────────────────────────────
# Prevents accidental re-runs — you must type 'yes' to start
confirm = input("Type 'yes' to start SRQ1 run (anything else cancels): ")
if confirm.strip().lower() == "yes":
    run_srq1()
else:
    print("Run cancelled.")

---

> ⚠️ **Do not run on a clean install.**

This cell issues API calls to (re-)run individual entries that failed during the main run. Only run it if your `results/*.jsonl` file is missing entries after a crash. The committed result files in `results/` are complete — running this cell on top of them does nothing useful but costs API quota.

Dont run these two blocks unless there is missing output from Gemini, Claude.

In [ ]:
with open("/str(RESULTS_DIR / "srq1_results_fixed.jsonl")") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

# Find all ERROR entries
all_errors = df[df["model_response"] == "ERROR"][
    ["piece_id", "model", "question_type"]
].reset_index(drop=True)

print(f"Total error entries to re-run: {len(all_errors)}")
print(all_errors.to_string())

In [ ]:
# ── RE-RUN ALL ERROR ENTRIES ───────────────────────────────────────────────
# Load current fixed results
with open("/str(RESULTS_DIR / "srq1_results_fixed.jsonl")") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

# Load ground truth lookup
with open(GROUND_TRUTH_FILE) as f:
    gt_data = json.load(f)
gt_lookup = {g["movement_id"]: g for g in gt_data}

# Get all error entries
error_entries = df[df["model_response"] == "ERROR"][
    ["piece_id", "model", "question_type"]
].values.tolist()

print(f"Re-running {len(error_entries)} error entries...")

# Model function mapping
model_fn_map = {
    "claude-sonnet-4-6": query_claude,
    "gemini-2.5-flash":  query_gemini,
    "gpt-5.4":           query_gpt
}

# Task prompt and parser mapping
task_prompts = {
    "key_signature":  prompt_srq1_key(),
    "time_signature": prompt_srq1_time()
}
task_parsers = {
    "key_signature":  parse_key_signature,
    "time_signature": parse_time_signature
}

# Re-run each error entry
new_entries = []
for i, (piece_id, model_name, q_type) in enumerate(error_entries):
    gt = gt_lookup.get(piece_id)
    if gt is None:
        print(f"  [SKIP] No ground truth for {piece_id}")
        continue

    ground_truth_val = gt.get(q_type)
    if ground_truth_val is None:
        print(f"  [SKIP] No ground truth for {piece_id} | {q_type}")
        continue

    pdf_path = PDF_DIR / f"{piece_id}.pdf"
    if not pdf_path.exists():
        print(f"  [SKIP] PDF not found: {piece_id}.pdf")
        continue

    query_fn  = model_fn_map[model_name]
    prompt_text = task_prompts[q_type]
    parser_fn   = task_parsers[q_type]

    raw_response = retry_call(query_fn, pdf_path, prompt_text)
    parsed       = parser_fn(raw_response)

    if q_type == "key_signature":
        correct = keys_match(ground_truth_val, parsed)
    else:
        correct = (
            parsed is not None and
            parsed.strip() == str(ground_truth_val).strip()
        )

    entry = {
        "piece_id":       piece_id,
        "model":          model_name,
        "prompt_variant": "zero_shot",
        "input_type":     "pdf",
        "question_type":  q_type,
        "ground_truth":   str(ground_truth_val),
        "model_response": raw_response,
        "parsed_answer":  parsed,
        "correct":        correct
    }

    new_entries.append(entry)
    print(f"  [{i+1}/{len(error_entries)}] {piece_id} | {model_name} | {q_type} | correct={correct}")

print(f"\nRe-run complete. Got {len(new_entries)} new entries.")

In [ ]:
# ── MERGE AND SAVE ────────────────────────────────────────────────────────────

# Build set of (piece_id, model, question_type) that were re-run
rerun_keys = {
    (e["piece_id"], e["model"], e["question_type"])
    for e in new_entries
}

# Remove old ERROR entries that were re-run
df_filtered = df[~(
    (df["model_response"] == "ERROR") &
    df.apply(
        lambda r: (r["piece_id"], r["model"], r["question_type"]) in rerun_keys,
        axis=1
    )
)]

# Add new entries
df_new    = pd.DataFrame(new_entries)
df_final  = pd.concat([df_filtered, df_new], ignore_index=True)
df_final  = df_final.sort_values(
    ["piece_id", "model", "question_type"]
).reset_index(drop=True)

# Save back to fixed file
df_final.to_json(
    "/str(RESULTS_DIR / "srq1_results_fixed.jsonl")",
    orient="records",
    lines=True
)

print(f"Saved {len(df_final)} entries to srq1_results_fixed.jsonl")
print(f"\nRemaining errors: {(df_final['model_response'] == 'ERROR').sum()}")
print(f"\nEntries per model:")
print(df_final["model"].value_counts().to_string())

## Part 8 — SRQ2: Key + Time Signature; CoT faithfulness + XAI (CoT, PDF)
### Cell 16 — Run SRQ2 in PDF format for all three models.

In [ ]:
SRQ2_RESULTS_FILE = RESULTS_DIR / "srq2_cot_results.jsonl"

def run_srq2_cot():
    """
    NEW SRQ2 — CoT Faithfulness Analysis.
    Runs CoT prompts on key + time signature identification
    across all movements and all three models.

    Each entry saves:
    - full model response (the reasoning + final answer)
    - extracted final answer (parsed)
    - extracted reasoning text (everything before 'Final answer:')
    - correctness flag

    Compare accuracy against SRQ1 zero-shot baseline for
    CoT vs zero-shot comparison.
    Manual faithfulness coding is done post-hoc on wrong-answer cases only.
    """
    with open(GROUND_TRUTH_FILE) as f:
        gt_data = json.load(f)

    completed = load_completed(SRQ2_RESULTS_FILE)

    models = {
        "claude-sonnet-4-6": query_claude,
        "gemini-2.5-flash":  query_gemini,
        "gpt-5.4":           query_gpt
    }

    tasks = [
        ("key_signature",  prompt_srq2_cot_key(),  "key_signature"),
        ("time_signature", prompt_srq2_cot_time(), "time_signature")
    ]

    total = len(gt_data) * len(models) * len(tasks)
    done  = len(completed)

    print(f"Total expected entries: {total}")
    print(f"Already completed:      {done}")
    print(f"Remaining:              {total - done}\n")

    for gt in gt_data:
        mid      = gt["movement_id"]
        pdf_path = PDF_DIR / gt.get("pdf_file", f"{mid}.pdf")

        if not pdf_path.exists():
            print(f"  [SKIP] PDF not found: {pdf_path.name}")
            continue

        for gt_field, prompt_text, q_type in tasks:
            ground_truth_val = gt.get(gt_field)
            if ground_truth_val is None:
                continue

            for model_name, query_fn in models.items():
                key = (mid, model_name, "cot", gt_field)
                if key in completed:
                    done += 1
                    continue

                # Get full CoT response
                raw_response = retry_call(query_fn, pdf_path, prompt_text)

                # Parse: extract final answer + reasoning separately
                parsed, reasoning = parse_cot_response(raw_response, q_type)

                # Correctness check
                if gt_field == "key_signature":
                    correct = keys_match(ground_truth_val, parsed)
                else:
                    correct = (
                        parsed is not None and
                        parsed.strip() == str(ground_truth_val).strip()
                    )

                entry = {
                    "piece_id":         mid,
                    "model":            model_name,
                    "prompt_variant":   "cot",
                    "input_type":       "pdf",
                    "question_type":    gt_field,
                    "ground_truth":     str(ground_truth_val),
                    "model_response":   raw_response,  # full CoT text
                    "reasoning":        reasoning,      # extracted reasoning only
                    "parsed_answer":    parsed,         # extracted final answer
                    "correct":          correct
                }

                save_result(SRQ2_RESULTS_FILE, entry)
                done += 1
                print(
                    f"  [{done}/{total}] {mid} | {model_name} | "
                    f"{gt_field} | correct={correct}"
                )

    print(f"\nSRQ2 CoT done. Results saved to {SRQ2_RESULTS_FILE}")


# ── SAFETY GUARD ──────────────────────────────────────────────────────────────
confirm = input("Type 'yes' to start SRQ2 CoT run (anything else cancels): ")
if confirm.strip().lower() == "yes":
    run_srq2_cot()
else:
    print("Run cancelled.")

In [ ]:
# ── REPARSE SRQ2 RESULTS (no API calls — uses saved raw responses) ────────────
# Run this after fixing parse_time_signature to recompute parsed_answer
# and the correct flag on the existing srq2_cot_results.jsonl.
# Writes the cleaned-up file to srq2_cot_results.jsonl (overwrites in place
# after writing to a temp file).

def reparse_srq2_results():
    src = RESULTS_DIR / "srq2_cot_results.jsonl"
    tmp = RESULTS_DIR / "srq2_cot_results.REPARSED.jsonl"

    if not src.exists():
        print(f"Source file not found: {src}")
        return

    n_total = 0
    n_changed = 0
    n_flipped_to_correct = 0
    n_flipped_to_wrong = 0

    with open(src) as f_in, open(tmp, "w") as f_out:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            n_total += 1

            old_parsed = entry.get("parsed_answer")
            old_correct = entry.get("correct")

            # Re-run the parser on the saved raw response
            new_parsed, new_reasoning = parse_cot_response(
                entry["model_response"],
                entry["question_type"]
            )

            # Recompute correctness with the same logic as the main runner
            gt_val = entry["ground_truth"]
            if entry["question_type"] == "key_signature":
                new_correct = keys_match(gt_val, new_parsed)
            else:
                new_correct = (
                    new_parsed is not None
                    and new_parsed.strip() == str(gt_val).strip()
                )

            if new_parsed != old_parsed or new_correct != old_correct:
                n_changed += 1
                if new_correct and not old_correct:
                    n_flipped_to_correct += 1
                    print(f"  FIXED → correct: {entry['piece_id']} | "
                          f"{entry['model']} | {entry['question_type']} | "
                          f"was {old_parsed!r}, now {new_parsed!r}")
                elif not new_correct and old_correct:
                    n_flipped_to_wrong += 1
                    print(f"  FLIPPED → wrong: {entry['piece_id']} | "
                          f"{entry['model']} | {entry['question_type']}")

            entry["parsed_answer"] = new_parsed
            entry["reasoning"] = new_reasoning
            entry["correct"] = new_correct
            f_out.write(json.dumps(entry) + "\n")

    # Replace original with reparsed version
    src.unlink()
    tmp.rename(src)

    print(f"\nReparsed {n_total} entries.")
    print(f"  Changed:                      {n_changed}")
    print(f"  Newly correct (was wrong):    {n_flipped_to_correct}")
    print(f"  Newly wrong (was correct):    {n_flipped_to_wrong}")
    print(f"\nResults file updated in place: {src}")
    print("Now regenerate the coding template with generate_coding_template().")

reparse_srq2_results()

### Manual coding CSV generator

In [ ]:
# ── GENERATE MANUAL CODING TEMPLATE ──────────────────────────────────────────
# Run after run_srq2_cot() completes.
# Creates a CSV pre-filled with wrong-answer CoT responses.
# You fill in the coding columns manually.

def generate_coding_template():
    """
    Load SRQ2 CoT results, filter to wrong-answer cases only,
    and export a CSV template for manual faithfulness coding.

    Coding columns (you fill these in):
    ─────────────────────────────────────────────────────────────────
    KEY SIGNATURE:
      accidental_count_correct  — did the model count sharps/flats correctly?
                                  (True / False / Cannot determine)
      accidental_type_correct   — did it identify sharp vs flat correctly?
                                  (True / False / Cannot determine)
      key_mapping_correct       — given its stated accidentals, did it map
                                  to the right key name?
                                  (True / False / Cannot determine)
      reasoning_category        — overall category:
        A = correct reasoning, wrong answer  (reasoning-answer mismatch)
        B = wrong reasoning, wrong answer    (consistent failure)
        C = unclear/incomplete reasoning     (cannot determine)

    TIME SIGNATURE:
      top_number_correct        — did the model identify beats per bar correctly?
                                  (True / False / Cannot determine)
      bottom_number_correct     — did it identify the note value correctly?
                                  (True / False / Cannot determine)
      symbol_described_correct  — did it correctly describe the visual symbol?
                                  (True / False / Cannot determine)
      reasoning_category        — same A/B/C as above
    ─────────────────────────────────────────────────────────────────
    """
    srq2_file = RESULTS_DIR / "srq2_cot_results.jsonl"

    if not srq2_file.exists():
        print("SRQ2 results not found. Run run_srq2_cot() first.")
        return

    records = []
    with open(srq2_file) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    df = pd.DataFrame(records)
    n_total  = len(df)
    n_errors = (df["model_response"] == "ERROR").sum()
    n_correct = df["correct"].sum()
    n_wrong  = (~df["correct"] & (df["model_response"] != "ERROR")).sum()

    print(f"Total entries:   {n_total}")
    print(f"Correct:         {n_correct} ({n_correct/n_total:.1%})")
    print(f"Wrong:           {n_wrong}   ({n_wrong/n_total:.1%})")
    print(f"Errors:          {n_errors}")
    print(f"\nManual coding needed for {n_wrong} wrong-answer cases.")

    # Filter to wrong answers only (exclude ERRORs)
    df_wrong = df[
        (~df["correct"]) &
        (df["model_response"] != "ERROR")
    ].copy()

    # Add empty coding columns
    df_wrong["accidental_count_correct"]  = ""
    df_wrong["accidental_type_correct"]   = ""
    df_wrong["key_mapping_correct"]       = ""
    df_wrong["top_number_correct"]        = ""
    df_wrong["bottom_number_correct"]     = ""
    df_wrong["symbol_described_correct"]  = ""
    df_wrong["reasoning_category"]        = ""  # A / B / C
    df_wrong["coder_notes"]               = ""  # free text

    # Sort for easier reading: by task, then model, then piece
    df_wrong = df_wrong.sort_values(
        ["question_type", "model", "piece_id"]
    ).reset_index(drop=True)

    # Select columns for the coding sheet
    # Keep reasoning visible so you can read it while coding
    output_cols = [
        "piece_id", "model", "question_type",
        "ground_truth", "parsed_answer",
        "reasoning",                    # read this to code
        "accidental_count_correct",     # fill in (key sig only)
        "accidental_type_correct",      # fill in (key sig only)
        "key_mapping_correct",          # fill in (key sig only)
        "top_number_correct",           # fill in (time sig only)
        "bottom_number_correct",        # fill in (time sig only)
        "symbol_described_correct",     # fill in (time sig only)
        "reasoning_category",           # fill in (A / B / C)
        "coder_notes"                   # optional free text
    ]

    out_path = RESULTS_DIR / "srq2_manual_coding_template.csv"
    df_wrong[output_cols].to_csv(out_path, index=False)

    print(f"\nCoding template saved to: {out_path}")
    print(f"Open in Excel or Google Sheets to fill in the coding columns.")
    print(f"\nCoding guide:")
    print(f"  A = model stated correct intermediate reasoning but gave wrong final answer")
    print(f"  B = model stated incorrect intermediate reasoning AND gave wrong final answer")
    print(f"  C = reasoning is too vague or unclear to determine")
    print(f"\nFor key signature: fill in columns 7-9 only.")
    print(f"For time signature: fill in columns 10-12 only.")
    print(f"Column 13 (reasoning_category) applies to both.")

    return df_wrong


coding_df = generate_coding_template()

In [ ]:
df = pd.read_csv(RESULTS_DIR / "srq2_manual_coding_template.csv")
print("Wrong-answer breakdown:")
print(df.groupby(["model", "question_type"]).size().unstack(fill_value=0))
print(f"\nTotal: {len(df)}")

---
## Part 9 — SRQ3: PDF vs PNG Input Format Comparison
### Cell 17 — Run SRQ3 across both input formats and all three models.

In [ ]:
SRQ3_RESULTS_FILE = RESULTS_DIR / "srq3_results.jsonl"

def run_srq3():
    """
    Run SRQ3: key + time signature identification, PDF vs PNG input format.
    - PDF: single file per movement
    - PNG: all pages for the movement sent together in one API call
    - Prompting: zero_shot_role (same for both formats)
    """
    with open(GROUND_TRUTH_FILE) as f:
        gt_data = json.load(f)

    completed = load_completed(SRQ3_RESULTS_FILE)

    pdf_models = {
        "claude-sonnet-4-6": query_claude,
        "gemini-2.5-flash":  query_gemini,
        "gpt-5.4":           query_gpt
    }
    png_models = {
        "claude-sonnet-4-6": query_claude_png,
        "gemini-2.5-flash":  query_gemini_png,
        "gpt-5.4":           query_gpt_png
    }

    tasks = [
        ("key_signature",  prompt_srq3_key(),  parse_key_signature),
        ("time_signature", prompt_srq3_time(), parse_time_signature)
    ]

    total = len(gt_data) * len(pdf_models) * len(tasks) * 2  # x2 for PDF + PNG
    done  = len(completed)

    print(f"Total expected entries: {total}")
    print(f"Already completed:      {done}")
    print(f"Remaining:              {total - done}\n")

    for gt in gt_data:
        mid = gt["movement_id"]

        # ── PDF input: single file ────────────────────────────────────────────
        pdf_path = PDF_DIR / gt.get("pdf_file", f"{mid}.pdf")

        # ── PNG input: all pages as a sorted list ─────────────────────────────
        png_paths = get_png_pages(mid)

        input_configs = [
            ("pdf", pdf_models, pdf_path),
            ("png", png_models, png_paths),
        ]

        for fmt, model_dict, file_input in input_configs:

            # Validate input exists
            if fmt == "pdf":
                if not pdf_path.exists():
                    print(f"  [SKIP] PDF not found: {pdf_path.name}")
                    continue
            else:  # png
                if not png_paths:
                    print(f"  [SKIP] No PNG pages found for: {mid}")
                    continue
                print(f"  [INFO] {mid} PNG pages: {len(png_paths)}")

            for model_name, query_fn in model_dict.items():
                for gt_field, prompt_text, parser_fn in tasks:

                    if gt[gt_field] is None:
                        continue

                    # prompt_variant includes format so resume key matches saved entry
                    prompt_variant_label = f"zero_shot_role_{fmt}"
                    key = (mid, model_name, prompt_variant_label, gt_field)
                    if key in completed:
                        done += 1
                        continue

                    raw_response     = retry_call(query_fn, file_input, prompt_text)
                    parsed           = parser_fn(raw_response)
                    ground_truth_val = gt[gt_field]

                    # Use keys_match() for key signatures (handles DCML format)
                    if gt_field == "key_signature":
                        correct = keys_match(ground_truth_val, parsed)
                    else:
                        correct = (
                            parsed is not None and
                            parsed.strip() == str(ground_truth_val).strip()
                        )

                    entry = {
                        "piece_id":       mid,
                        "model":          model_name,
                        "prompt_variant": prompt_variant_label,
                        "input_type":     fmt,
                        "question_type":  gt_field,
                        "ground_truth":   str(ground_truth_val),
                        "model_response": raw_response,
                        "parsed_answer":  parsed,
                        "correct":        correct
                    }

                    save_result(SRQ3_RESULTS_FILE, entry)
                    done += 1
                    print(f"  [{done}/{total}] {mid} | {model_name} | {fmt} | {gt_field} | correct={correct}")

    print(f"\nSRQ3 done. Results saved to {SRQ3_RESULTS_FILE}")


# ── SAFETY GUARD ──────────────────────────────────────────────────────────────
confirm = input("Type 'yes' to start SRQ3 run (anything else cancels): ")
if confirm.strip().lower() == "yes":
    run_srq3()
else:
    print("Run cancelled.")

---
### Only run these if there's ERROR

In [ ]:
# ── INSPECT SRQ3 ERRORS ───────────────────────────────────────────────────────

with open(SRQ3_RESULTS_FILE) as f:
    srq3_records = [json.loads(line) for line in f if line.strip()]

df_srq3 = pd.DataFrame(srq3_records)

all_errors = df_srq3[df_srq3["model_response"] == "ERROR"][
    ["piece_id", "model", "input_type", "question_type", "prompt_variant"]
].reset_index(drop=True)

print(f"Total entries:        {len(df_srq3)}")
print(f"Error entries:        {len(all_errors)}")
print(f"Error rate:           {len(all_errors)/len(df_srq3):.1%}")
print(f"\nError breakdown:\n{all_errors.to_string()}")

In [ ]:
# ── RERUN SRQ3 ERROR ENTRIES ──────────────────────────────────────────────────

# Load ground truth lookup
with open(GROUND_TRUTH_FILE) as f:
    gt_data = json.load(f)
gt_lookup = {g["movement_id"]: g for g in gt_data}

# Model function mapping (single file input)
pdf_model_map = {
    "claude-sonnet-4-6": query_claude,
    "gemini-2.5-flash":  query_gemini,
    "gpt-5.4":           query_gpt
}

# Model function mapping (multi-page PNG input)
png_model_map = {
    "claude-sonnet-4-6": query_claude_png,
    "gemini-2.5-flash":  query_gemini_png,
    "gpt-5.4":           query_gpt_png
}

# Task prompt and parser mapping
task_prompts = {
    "key_signature":  prompt_srq3_key(),
    "time_signature": prompt_srq3_time()
}
task_parsers = {
    "key_signature":  parse_key_signature,
    "time_signature": parse_time_signature
}

# Get error entries as list of tuples
error_entries = df_srq3[df_srq3["model_response"] == "ERROR"][
    ["piece_id", "model", "input_type", "question_type"]
].values.tolist()

print(f"Re-running {len(error_entries)} error entries...\n")

new_entries = []

for i, (piece_id, model_name, input_type, q_type) in enumerate(error_entries):
    gt = gt_lookup.get(piece_id)
    if gt is None:
        print(f"  [SKIP] No ground truth for {piece_id}")
        continue

    ground_truth_val = gt.get(q_type)
    if ground_truth_val is None:
        print(f"  [SKIP] No ground truth for {piece_id} | {q_type}")
        continue

    # ── Get file input based on format ────────────────────────────────────────
    if input_type == "pdf":
        file_input  = PDF_DIR / gt.get("pdf_file", f"{piece_id}.pdf")
        query_fn    = pdf_model_map[model_name]
        if not file_input.exists():
            print(f"  [SKIP] PDF not found: {file_input.name}")
            continue

    else:  # png
        file_input  = get_png_pages(piece_id)
        query_fn    = png_model_map[model_name]
        if not file_input:
            print(f"  [SKIP] No PNG pages found for: {piece_id}")
            continue

    prompt_text      = task_prompts[q_type]
    parser_fn        = task_parsers[q_type]
    prompt_variant   = f"zero_shot_role_{input_type}"

    raw_response     = retry_call(query_fn, file_input, prompt_text)
    parsed           = parser_fn(raw_response)

    # Correctness check
    if q_type == "key_signature":
        correct = keys_match(ground_truth_val, parsed)
    else:
        correct = (
            parsed is not None and
            parsed.strip() == str(ground_truth_val).strip()
        )

    entry = {
        "piece_id":       piece_id,
        "model":          model_name,
        "prompt_variant": prompt_variant,
        "input_type":     input_type,
        "question_type":  q_type,
        "ground_truth":   str(ground_truth_val),
        "model_response": raw_response,
        "parsed_answer":  parsed,
        "correct":        correct
    }

    new_entries.append(entry)
    print(f"  [{i+1}/{len(error_entries)}] {piece_id} | {model_name} | {input_type} | {q_type} | correct={correct}")

print(f"\nRe-run complete. Got {len(new_entries)} new entries.")

In [ ]:
# ── MERGE AND SAVE ────────────────────────────────────────────────────────────

# Keys of entries that were re-run
rerun_keys = {
    (e["piece_id"], e["model"], e["input_type"], e["question_type"])
    for e in new_entries
}

# Remove old ERROR entries that were re-run
df_filtered = df_srq3[~(
    (df_srq3["model_response"] == "ERROR") &
    df_srq3.apply(
        lambda r: (r["piece_id"], r["model"], r["input_type"], r["question_type"]) in rerun_keys,
        axis=1
    )
)]

# Add new entries
df_new   = pd.DataFrame(new_entries)
df_final = pd.concat([df_filtered, df_new], ignore_index=True)
df_final = df_final.sort_values(
    ["piece_id", "model", "input_type", "question_type"]
).reset_index(drop=True)

# Save back
df_final.to_json(SRQ3_RESULTS_FILE, orient="records", lines=True)

print(f"Saved {len(df_final)} entries to {SRQ3_RESULTS_FILE.name}")
print(f"Remaining errors: {(df_final['model_response'] == 'ERROR').sum()}")
print(f"\nBreakdown by format + model:")
print(df_final.groupby(["input_type", "model"]).size().to_string())

---
## Part 10 — Quick Results Summary

Check class imbalance:

In [ ]:
with open("/str(RESULTS_DIR / "srq1_results_fixed.jsonl")") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

key_df = df[df["question_type"] == "key_signature"]
print("Key signature class distribution:")
print(key_df["ground_truth"].value_counts().to_string())
print(f"\nUnique classes: {key_df['ground_truth'].nunique()}")
print(f"Total samples: {len(key_df)}")

In [ ]:
# ── LOAD DATA ─────────────────────────────────────────────────────────────────

with open("/str(RESULTS_DIR / "srq1_results_fixed.jsonl")") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

# Use only one model to avoid triple-counting
# (all models see the same ground truth)
df_single = df[df["model"] == "claude-sonnet-4-6"].copy()

key_df  = df_single[df_single["question_type"] == "key_signature"]
time_df = df_single[df_single["question_type"] == "time_signature"]

# ── NORMALISE KEY SIGNATURE LABELS FOR DISPLAY ────────────────────────────────
# Convert DCML format to readable labels e.g. 'f' → 'F minor', 'C' → 'C major'

def dcml_to_display(dcml_key):
    if not dcml_key:
        return dcml_key
    quality = "major" if dcml_key[0].isupper() else "minor"
    note = dcml_key[0].upper() + dcml_key[1:]
    return f"{note} {quality}"

key_counts  = key_df["ground_truth"].value_counts()
key_labels  = [dcml_to_display(k) for k in key_counts.index]
key_values  = key_counts.values

time_counts = time_df["ground_truth"].value_counts()
time_labels = time_counts.index.tolist()
time_values = time_counts.values

# ── COLOUR CODING ─────────────────────────────────────────────────────────────
# Major keys = blue tones, Minor keys = orange tones

def get_key_colour(dcml_key):
    return "#4C72B0" if dcml_key[0].isupper() else "#DD8452"

key_colours  = [get_key_colour(k) for k in key_counts.index]
time_colours = ["#55A868"] * len(time_counts)

# ── PLOT ──────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Ground Truth Class Distribution — SRQ1 Sample",
    fontsize=14, fontweight="bold", y=1.02
)

# ── LEFT: Key Signature ───────────────────────────────────────────────────────
ax1 = axes[0]
bars1 = ax1.barh(key_labels, key_values, color=key_colours, edgecolor="white")
ax1.set_xlabel("Number of movements", fontsize=11)
ax1.set_title("Key Signature Distribution", fontsize=12, fontweight="bold")
ax1.invert_yaxis()  # Most frequent at top

# Add count labels inside bars
for bar, val in zip(bars1, key_values):
    ax1.text(
        bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
        str(val), va="center", ha="left", fontsize=9
    )

# Legend
major_patch = mpatches.Patch(color="#4C72B0", label="Major")
minor_patch = mpatches.Patch(color="#DD8452", label="Minor")
ax1.legend(handles=[major_patch, minor_patch], loc="lower right", fontsize=10)
ax1.set_xlim(0, key_values.max() + 3)

# ── RIGHT: Time Signature ─────────────────────────────────────────────────────
ax2 = axes[1]
bars2 = ax2.bar(time_labels, time_values, color=time_colours, edgecolor="white")
ax2.set_xlabel("Time signature", fontsize=11)
ax2.set_ylabel("Number of movements", fontsize=11)
ax2.set_title("Time Signature Distribution", fontsize=12, fontweight="bold")

# Add count labels on top of bars
for bar, val in zip(bars2, time_values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        str(val), va="bottom", ha="center", fontsize=10, fontweight="bold"
    )

ax2.set_ylim(0, time_values.max() + 5)

plt.tight_layout()
plt.savefig(
    "str(RESULTS_DIR / "class_distribution.png")",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Plot saved to results/class_distribution.png")

What the distribution tells us:
- 16 unique classes
- Range from 24 (C major) down to 3 (Bb, g, Db)
- No class appears only once — minimum is 3
- The ratio between most and least frequent is 24/3 = 8x

So, use macro-average is okay.

---

### Cell 18 — Print all ML evaluation metrics for all sub-research questions. (with macro-average)

In [ ]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

# ── CORE LOADER ───────────────────────────────────────────────────────────────

def load_results(results_file):
    """Load JSONL results file into a pandas DataFrame."""
    if not Path(results_file).exists():
        print(f"No results file found at {results_file}")
        return None

    records = []
    with open(results_file) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} entries from {Path(results_file).name}")
    print(f"Error rate: {(df['model_response'] == 'ERROR').mean():.1%}")
    return df

# ── NORMALISER ────────────────────────────────────────────────────────────────

def normalise_for_sklearn(val, question_type):
    """
    Normalise ground truth and parsed answers to the same string format
    before passing to sklearn metrics.
    For key_signature: convert everything to 'note quality' format
      e.g. 'f' → 'f minor', 'C' → 'c major', 'f minor' → 'f minor'
    For other tasks: just lowercase and strip.
    """
    if not val or str(val).strip() == "":
        return "unknown"

    val = str(val).strip()

    if question_type == "key_signature":
        # DCML format (short, no spaces) e.g. 'f', 'C', 'Bb'
        if len(val) <= 3 and " " not in val:
            note, quality = dcml_key_to_standard(val)
            if note and quality:
                return f"{note} {quality}"
        # Model response format e.g. 'F minor', 'B-flat major'
        note, quality = normalise_key_for_comparison(val)
        if note and quality:
            return f"{note} {quality}"
        return val.lower()

    return val.lower().strip()

# ── METRIC CALCULATOR ─────────────────────────────────────────────────────────

def compute_metrics(df_subset, question_type=None):
    """
    Compute accuracy, macro precision, macro recall, macro F1
    for a subset of results.

    Uses macro averaging for all task types — each class gets
    equal weight regardless of frequency. Justified by the moderate
    class balance in the dataset (16 key classes, min 3 samples each).

    Accuracy is computed from the pre-computed correct column which
    uses keys_match() for key signature and exact string match for
    time signature and pitch.
    """
    df_clean = df_subset[df_subset["model_response"] != "ERROR"].copy()

    if df_clean.empty:
        return {
            "n":        0,
            "n_errors": len(df_subset),
            "accuracy": None,
            "precision": None,
            "recall":   None,
            "f1":       None,
        }

    # Normalise to same string format before sklearn comparison
    y_true = df_clean["ground_truth"].apply(
        lambda x: normalise_for_sklearn(x, question_type)
    )
    y_pred = df_clean["parsed_answer"].fillna("").apply(
        lambda x: normalise_for_sklearn(x, question_type)
    )

    # Accuracy from pre-computed correct column
    accuracy = df_clean["correct"].mean()

    # Macro averaging — equal weight per class
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "n":        len(df_clean),
        "n_errors": len(df_subset) - len(df_clean),
        "accuracy":  round(accuracy, 3),
        "precision": round(precision, 3),
        "recall":    round(recall, 3),
        "f1":        round(f1, 3),
    }

# ── SRQ1 EVALUATION ───────────────────────────────────────────────────────────

def evaluate_srq1(results_file=None):
    """
    SRQ1: Key + time signature identification (zero-shot, PDF).
    Groups by model × question_type.
    """
    if results_file is None:
        results_file = SRQ1_RESULTS_FILE

    df = load_results(results_file)
    if df is None:
        return

    print("\n" + "="*60)
    print("SRQ1 EVALUATION — Key + Time Signature (Zero-shot, PDF)")
    print("="*60)

    rows = []
    for (model, q_type), group in df.groupby(["model", "question_type"]):
        metrics = compute_metrics(group, question_type=q_type)
        rows.append({
            "model":     model,
            "task":      q_type,
            "n":         metrics["n"],
            "errors":    metrics["n_errors"],
            "accuracy":  metrics["accuracy"],
            "precision": metrics["precision"],
            "recall":    metrics["recall"],
            "f1":        metrics["f1"],
        })

    results_df = pd.DataFrame(rows).sort_values(["task", "model"])
    print(results_df.to_string(index=False))
    print("\nAll metrics use macro averaging.")
    return results_df

# ── SRQ2 EVALUATION (CoT Faithfulness) ────────────────────────────────────────

def evaluate_srq2(results_file=None):
    """
    SRQ2: Key + time signature identification with Chain-of-Thought prompting.
    Groups by model × question_type.

    Reports CoT accuracy. The faithfulness analysis itself is qualitative
    and comes from the separate manual coding step.
    """
    if results_file is None:
        results_file = SRQ2_RESULTS_FILE

    df = load_results(results_file)
    if df is None:
        return

    print("\n" + "="*60)
    print("SRQ2 EVALUATION — Key + Time Signature (CoT, PDF)")
    print("="*60)

    rows = []
    for (model, q_type), group in df.groupby(["model", "question_type"]):
        metrics = compute_metrics(group, question_type=q_type)
        rows.append({
            "model":     model,
            "task":      q_type,
            "n":         metrics["n"],
            "errors":    metrics["n_errors"],
            "accuracy":  metrics["accuracy"],
            "precision": metrics["precision"],
            "recall":    metrics["recall"],
            "f1":        metrics["f1"],
        })

    results_df = pd.DataFrame(rows).sort_values(["task", "model"])
    print(results_df.to_string(index=False))
    print("\nAll metrics use macro averaging.")
    return results_df


# ── SRQ1 vs SRQ2 COMPARISON (zero-shot vs CoT) ────────────────────────────────

def compare_srq1_srq2(srq1_file=None, srq2_file=None):
    """
    Compare zero-shot (SRQ1) vs CoT (SRQ2) accuracy on the same tasks.
    This is the headline 'does CoT help?' result.
    """
    if srq1_file is None:
        srq1_file = SRQ1_RESULTS_FILE
    if srq2_file is None:
        srq2_file = SRQ2_RESULTS_FILE

    df1 = load_results(srq1_file)
    df2 = load_results(srq2_file)
    if df1 is None or df2 is None:
        return

    print("\n" + "="*60)
    print("SRQ1 vs SRQ2 — Zero-shot vs CoT accuracy")
    print("="*60)

    rows = []
    for (model, q_type), g1 in df1.groupby(["model", "question_type"]):
        g2 = df2[(df2["model"] == model) & (df2["question_type"] == q_type)]
        if g2.empty:
            continue

        m1 = compute_metrics(g1, question_type=q_type)
        m2 = compute_metrics(g2, question_type=q_type)

        delta = (m2["accuracy"] - m1["accuracy"]) \
                if (m1["accuracy"] is not None and m2["accuracy"] is not None) \
                else None

        rows.append({
            "model":         model,
            "task":          q_type,
            "zero_shot_acc": m1["accuracy"],
            "cot_acc":       m2["accuracy"],
            "delta":         round(delta, 3) if delta is not None else None,
        })

    results_df = pd.DataFrame(rows).sort_values(["task", "model"])
    print(results_df.to_string(index=False))
    print("\nPositive delta = CoT helped. Negative = CoT hurt.")
    return results_df

# ── SRQ3 EVALUATION ───────────────────────────────────────────────────────────

def compute_consistency(df, model):
    """
    For SRQ3: proportion of movements where a model gives the same
    answer in both PDF and PNG formats, regardless of correctness.
    """
    model_df = df[df["model"] == model].copy()

    pdf_df = model_df[model_df["input_type"] == "pdf"][
        ["piece_id", "question_type", "parsed_answer"]
    ].rename(columns={"parsed_answer": "pdf_answer"})

    png_df = model_df[model_df["input_type"] == "png"][
        ["piece_id", "question_type", "parsed_answer"]
    ].rename(columns={"parsed_answer": "png_answer"})

    merged = pdf_df.merge(png_df, on=["piece_id", "question_type"])

    if merged.empty:
        return None

    merged["consistent"] = (
        merged["pdf_answer"].fillna("").str.strip().str.lower() ==
        merged["png_answer"].fillna("").str.strip().str.lower()
    )

    return round(merged["consistent"].mean(), 3)

def evaluate_srq3(results_file=None):
    """
    SRQ3: Key + time signature across PDF vs PNG input formats.
    Groups by model × input_type × question_type.
    Also reports consistency across formats.
    """
    if results_file is None:
        results_file = SRQ3_RESULTS_FILE

    df = load_results(results_file)
    if df is None:
        return

    print("\n" + "="*60)
    print("SRQ3 EVALUATION — PDF vs PNG Input Format Comparison")
    print("="*60)

    rows = []
    for (model, input_type, q_type), group in df.groupby(
        ["model", "input_type", "question_type"]
    ):
        metrics = compute_metrics(group, question_type=q_type)
        rows.append({
            "model":      model,
            "input_type": input_type,
            "task":       q_type,
            "n":          metrics["n"],
            "errors":     metrics["n_errors"],
            "accuracy":   metrics["accuracy"],
            "precision":  metrics["precision"],
            "recall":     metrics["recall"],
            "f1":         metrics["f1"],
        })

    results_df = pd.DataFrame(rows).sort_values(
        ["task", "input_type", "model"]
    )
    print(results_df.to_string(index=False))
    print("\nAll metrics use macro averaging.")

    # Consistency scores
    print("\n" + "-"*60)
    print("CONSISTENCY (same answer in PDF and PNG, regardless of correctness)")
    print("-"*60)
    for model in df["model"].unique():
        consistency = compute_consistency(df, model)
        if consistency is not None:
            print(f"  {model}: {consistency:.1%}")
        else:
            print(f"  {model}: insufficient data")

    return results_df

# ── COMBINED SUMMARY ──────────────────────────────────────────────────────────
def evaluate_all():
    """Run all three SRQ evaluations in sequence."""
    evaluate_srq1()
    evaluate_srq2()
    compare_srq1_srq2()
    evaluate_srq3()

print("Evaluation functions defined.")
print("Run:")
print("  evaluate_srq1()      — after SRQ1 run")
print("  evaluate_srq2()      — after SRQ2 run (CoT accuracy)")
print("  compare_srq1_srq2()  — zero-shot vs CoT delta")
print("  evaluate_srq3()      — after SRQ3 run")
print("  evaluate_all()       — all quantitative analyses")

### Don't run this cell until the outputs are shown.

In [ ]:
# After SRQ1
evaluate_srq1()

Seems like accuracy for time signature in Gemini is very bad, check if it's real and investigate the reasons. Plus, precision, recall, f1-score for key signature are near to 0 --> too many class, hence we switch the Macro-average to weighted-average.

In [ ]:
with open("/str(RESULTS_DIR / "srq1_results_fixed.jsonl")") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

# Look at Gemini time signature wrong answers
gemini_wrong = df[
    (df["model"] == "gemini-2.5-flash") &
    (df["question_type"] == "time_signature") &
    (df["correct"] == False) &
    (df["model_response"] != "ERROR")
]

print(f"Gemini wrong time signature answers: {len(gemini_wrong)}")
print("\nGround truth vs parsed answer:")
print(gemini_wrong[["piece_id", "ground_truth", "parsed_answer", "model_response"]].head(20).to_string())

Findings: Gemini 2.5 Flash showed notably lower accuracy on time signature identification (22.0%) compared to Claude Sonnet 4.6 (96.3%) and GPT-5.4 (87.8%). Qualitative inspection of incorrect responses revealed a systematic pattern of confusion between visually similar time signatures, suggesting limitations in fine-grained visual symbol recognition rather than a general failure to understand the task format.

---

In [ ]:
# After SRQ2
evaluate_srq2()

---

In [ ]:
# After SRQ3
evaluate_srq3()

---

In [ ]:
# After everything
evaluate_all()

---
## Additional metrics for SRQ1: Output Consistency across all models.

What this computes: For each (movement, task) pair in SRQ1, check if all 3 models gave the same parsed answer.

Report:
1. Overall agreement rate — % of (movement, task) pairs where all 3 models agreed
2. Per-task agreement rate — separately for key signature and time signature
3. Agreement breakdown table — so you can see which movements caused disagreement

In [ ]:
def evaluate_srq1_agreement():
    """
    SRQ1 additional metric: cross-model agreement (Definition A).
    For each (movement, task) pair, check if all 3 models gave the same answer.
    
    A triple is 'consistent' only if all 3 models agree exactly (all-or-nothing).
    Uses normalised parsed_answer for comparison (same normalisation as accuracy).
    """
    df = load_results(SRQ1_RESULTS_FILE)
    if df is None:
        return

    MODELS = ["claude-sonnet-4-6", "gemini-2.5-flash", "gpt-5.4"]
    TASKS  = ["key_signature", "time_signature"]

    # ── NORMALISE ANSWERS ─────────────────────────────────────────────────────
    # Use the same normaliser as the accuracy computation so comparisons are fair.
    # e.g. "F minor" and "f minor" should count as agreeing.

    df["answer_normalised"] = df.apply(
        lambda row: normalise_for_sklearn(row["parsed_answer"], row["question_type"]),
        axis=1
    )

    # ── BUILD AGREEMENT TABLE ─────────────────────────────────────────────────

    records = []

    for task in TASKS:
        df_task = df[df["question_type"] == task]
        movements = df_task["piece_id"].unique()

        for mid in movements:
            df_mov = df_task[df_task["piece_id"] == mid]

            # Get one answer per model — skip if a model is missing or errored
            model_answers = {}
            for model in MODELS:
                row = df_mov[df_mov["model"] == model]
                if row.empty:
                    continue
                resp = row.iloc[0]["model_response"]
                if resp == "ERROR":
                    continue
                model_answers[model] = row.iloc[0]["answer_normalised"]

            # Only evaluate if all 3 models have a valid answer
            if len(model_answers) < 3:
                continue

            answers = list(model_answers.values())
            all_agree = len(set(answers)) == 1

            records.append({
                "piece_id":       mid,
                "question_type":  task,
                "claude_answer":  model_answers.get("claude-sonnet-4-6", "missing"),
                "gemini_answer":  model_answers.get("gemini-2.5-flash",  "missing"),
                "gpt_answer":     model_answers.get("gpt-5.4",           "missing"),
                "all_agree":      all_agree
            })

    agreement_df = pd.DataFrame(records)

    # ── PRINT SUMMARY ─────────────────────────────────────────────────────────

    print("=" * 60)
    print("SRQ1 — Cross-Model Agreement (Definition A)")
    print("=" * 60)

    print("\nPer-task agreement rate (all 3 models give same answer):")
    for task in TASKS:
        sub = agreement_df[agreement_df["question_type"] == task]
        n   = len(sub)
        n_a = sub["all_agree"].sum()
        print(f"  {task:20s}: {n_a}/{n} = {n_a/n:.2%}")

    # Overall as a secondary summary only
    total   = len(agreement_df)
    n_agree = agreement_df["all_agree"].sum()
    print(f"\nCombined (both tasks): {n_agree}/{total} = {n_agree/total:.2%}")
    print("  (Note: combined figure is for reference only — tasks differ in difficulty)")

    # ── DISAGREEMENT CASES ────────────────────────────────────────────────────

    disagreements = agreement_df[~agreement_df["all_agree"]]
    print(f"\nDisagreement cases: {len(disagreements)}")

    if len(disagreements) > 0:
        print("\nSample disagreements (first 10):")
        print(
            disagreements[["piece_id", "question_type",
                           "claude_answer", "gemini_answer", "gpt_answer"]]
            .head(10)
            .to_string(index=False)
        )

    # ── SAVE ─────────────────────────────────────────────────────────────────

    out_path = RESULTS_DIR / "srq1_agreement.csv"
    agreement_df.to_csv(out_path, index=False)
    print(f"\nFull agreement table saved to: {out_path}")

    return agreement_df

In [ ]:
srq1_agreement_df = evaluate_srq1_agreement()

---
## Test (Run This First)
### Cell 19 — SRQ1 Quick sanity check: one PDF, one question, one model.
Run this before the full pipeline to confirm all three APIs are working.

In [ ]:
# Pick one movement that has a PDF
TEST_MOVEMENT = "01-1"
test_pdf = PDF_DIR / f"{TEST_MOVEMENT}.pdf"

assert test_pdf.exists(), f"PDF not found: {test_pdf}"

test_question = prompt_srq1_time()
print(f"Test question: {test_question}\n")

print("Testing Claude...")
r_claude = query_claude(test_pdf, test_question)
print(f"  Raw:    {r_claude}")
print(f"  Parsed: {parse_time_signature(r_claude)}\n")

print("Testing Gemini...")
r_gemini = query_gemini(test_pdf, test_question)
print(f"  Raw:    {r_gemini}")
print(f"  Parsed: {parse_time_signature(r_gemini)}\n")

print("Testing GPT-5.4...")
r_gpt = query_gpt(test_pdf, test_question)
print(f"  Raw:    {r_gpt}")
print(f"  Parsed: {parse_time_signature(r_gpt)}\n")

# Ground truth for 01-1 should be 2/2
print("Expected time signature for 01-1: 2/2")

### Test before SRQ3

In [ ]:
# Sanity check: verify PNG files exist for a few movements
sample_ids = ["01-1-1", "01-2-1", "02-1-1"]
for mid in sample_ids:
    p = PNG_DIR / f"{mid}.png"
    print(f"{mid}.png — exists: {p.exists()}")

In [ ]:
# Check 1: PNG pages load correctly for a few movements
for mid in ["01-1", "01-2", "02-1"]:
    pages = get_png_pages(mid)
    print(f"{mid}: {len(pages)} pages → {[p.name for p in pages]}")

In [ ]:
# Check 2: Test one PNG call per model (1 movement, time signature only)
test_pages = get_png_pages("01-1")
test_q = prompt_srq3_time()

print("Claude PNG:", query_claude_png(test_pages, test_q))
print("Gemini PNG:", query_gemini_png(test_pages, test_q))
print("GPT PNG:   ", query_gpt_png(test_pages, test_q))
# Expected: all should return "2/2"

In [ ]:
# Check 3: Expected entry count
with open(GROUND_TRUTH_FILE) as f:
    gt_data = json.load(f)
print(f"Movements: {len(gt_data)}")
print(f"Expected SRQ3 entries: {len(gt_data)} × 3 models × 2 tasks × 2 formats = {len(gt_data)*3*2*2}")

### Test run for SRQ2

In [ ]:
# ── SRQ2 SMOKE TEST — run on a single movement before the real run ───────────
# Verifies: API keys work, prompts return sensible output,
# CoT parser extracts the final answer correctly, JSONL writes cleanly.
# Writes to a separate test file so it doesn't pollute the real results.

from pathlib import Path

SRQ2_TEST_FILE = RESULTS_DIR / "srq2_cot_TEST.jsonl"

# Clear any previous test output so each smoke test starts clean
if SRQ2_TEST_FILE.exists():
    SRQ2_TEST_FILE.unlink()
    print(f"Cleared previous test file: {SRQ2_TEST_FILE.name}\n")

def smoke_test_srq2(movement_id="01-1"):
    """Run SRQ2 CoT on one movement across all 3 models and both tasks."""
    with open(GROUND_TRUTH_FILE) as f:
        gt_data = json.load(f)

    # Find the test movement
    gt = next((g for g in gt_data if g["movement_id"] == movement_id), None)
    if gt is None:
        print(f"Movement {movement_id} not found in ground truth.")
        return

    pdf_path = PDF_DIR / gt.get("pdf_file", f"{movement_id}.pdf")
    if not pdf_path.exists():
        print(f"PDF not found: {pdf_path}")
        return

    models = {
        "claude-sonnet-4-6": query_claude,
        "gemini-2.5-flash":  query_gemini,
        "gpt-5.4":           query_gpt,
    }

    tasks = [
        ("key_signature",  prompt_srq2_cot_key(),  "key_signature"),
        ("time_signature", prompt_srq2_cot_time(), "time_signature"),
    ]

    print(f"Smoke test on movement: {movement_id}")
    print(f"Ground truth — key: {gt.get('key_signature')}, "
          f"time: {gt.get('time_signature')}\n")

    for gt_field, prompt_text, q_type in tasks:
        ground_truth_val = gt.get(gt_field)
        if ground_truth_val is None:
            continue

        for model_name, query_fn in models.items():
            print(f"── {model_name} | {gt_field} ──")
            raw_response = retry_call(query_fn, pdf_path, prompt_text)
            parsed, reasoning = parse_cot_response(raw_response, q_type)

            if gt_field == "key_signature":
                correct = keys_match(ground_truth_val, parsed)
            else:
                correct = (
                    parsed is not None
                    and parsed.strip() == str(ground_truth_val).strip()
                )

            entry = {
                "piece_id":       movement_id,
                "model":          model_name,
                "prompt_variant": "cot",
                "input_type":     "pdf",
                "question_type":  gt_field,
                "ground_truth":   str(ground_truth_val),
                "model_response": raw_response,
                "reasoning":      reasoning,
                "parsed_answer":  parsed,
                "correct":        correct,
            }
            save_result(SRQ2_TEST_FILE, entry)

            # Show enough to spot problems
            print(f"  Raw (first 200 chars): {raw_response[:200]!r}")
            print(f"  Parsed answer:         {parsed!r}")
            print(f"  Ground truth:          {ground_truth_val!r}")
            print(f"  Correct:               {correct}\n")

    print(f"Test results saved to: {SRQ2_TEST_FILE}")
    print("If everything looks sensible, run the real pipeline in the next cell.")

smoke_test_srq2(movement_id="01-1")

---

Items all three models got wrong

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ANALYSIS CELL — Items all three models got wrong
# Works with SRQ1, SRQ2, or SRQ3 results files.
# ═══════════════════════════════════════════════════════════════════════════════

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Swap this to SRQ2_RESULTS_FILE or SRQ3_RESULTS_FILE as needed.
# For SRQ3: set FILTER_INPUT_TYPE to "pdf", "png", or None (to include both).

ANALYSIS_FILE     = SRQ1_RESULTS_FILE   # ← change me
FILTER_INPUT_TYPE = None                # ← "pdf" | "png" | None
SAVE_CSV          = True                # ← save the wrong-by-all table to CSV

MODEL_COLS = {
    "claude-sonnet-4-6": "claude_correct",
    "gemini-2.5-flash":  "gemini_correct",
    "gpt-5.4":           "gpt_correct",
}
ANSWER_COLS = {
    "claude-sonnet-4-6": "claude_answer",
    "gemini-2.5-flash":  "gemini_answer",
    "gpt-5.4":           "gpt_answer",
}

# ── LOAD ──────────────────────────────────────────────────────────────────────
with open(ANALYSIS_FILE) as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)
print(f"Loaded {len(df)} entries from {Path(ANALYSIS_FILE).name}")

# Optional: restrict to one input format (SRQ3 only)
if FILTER_INPUT_TYPE and "input_type" in df.columns:
    df = df[df["input_type"] == FILTER_INPUT_TYPE]
    print(f"Filtered to input_type='{FILTER_INPUT_TYPE}': {len(df)} entries")

# Drop API errors — a model that crashed is not a model that "got it wrong"
n_errors = (df["model_response"] == "ERROR").sum()
df_clean = df[df["model_response"] != "ERROR"].copy()
print(f"Dropped {n_errors} ERROR entries → {len(df_clean)} usable entries\n")

# ── SANITY CHECK — expected models present ────────────────────────────────────
found_models = df_clean["model"].unique().tolist()
missing = [m for m in MODEL_COLS if m not in found_models]
if missing:
    print(f"[WARNING] These models are missing from the file: {missing}")
    print(f"  Found: {found_models}")
    print("  Proceeding with available models only.\n")
    MODEL_COLS  = {k: v for k, v in MODEL_COLS.items()  if k in found_models}
    ANSWER_COLS = {k: v for k, v in ANSWER_COLS.items() if k in found_models}

# ── BUILD PIVOT TABLE ─────────────────────────────────────────────────────────
# Group key: (piece_id, question_type) — add input_type if SRQ3
group_keys = ["piece_id", "question_type"]
if "input_type" in df_clean.columns and df_clean["input_type"].nunique() > 1:
    group_keys.append("input_type")

# One row per (piece_id, question_type [, input_type]) per model
pivot_correct = (
    df_clean
    .groupby(group_keys + ["model"])["correct"]
    .first()
    .unstack("model")
    .rename(columns=MODEL_COLS)
    .reset_index()
)

pivot_answers = (
    df_clean
    .groupby(group_keys + ["model"])["parsed_answer"]
    .first()
    .unstack("model")
    .rename(columns=ANSWER_COLS)
    .reset_index()
)

# Ground truth (same for all models — just take first occurrence)
gt_df = (
    df_clean
    .groupby(group_keys)["ground_truth"]
    .first()
    .reset_index()
)

# Merge everything into one wide table
wide = pivot_correct.merge(pivot_answers, on=group_keys)
wide = wide.merge(gt_df, on=group_keys)

# ── FLAG: ALL THREE WRONG ─────────────────────────────────────────────────────
correct_cols = list(MODEL_COLS.values())

# A row counts as "all wrong" only if every model has a valid False entry.
# Rows where any model value is NaN are excluded (missing data, not a wrong answer).
has_all_models = wide[correct_cols].notna().all(axis=1)
all_wrong_mask = has_all_models & (wide[correct_cols] == False).all(axis=1)

all_wrong = wide[all_wrong_mask].copy().reset_index(drop=True)

# ── PRINT SUMMARY ─────────────────────────────────────────────────────────────
total_pairs   = len(wide[has_all_models])
n_all_wrong   = len(all_wrong)

print("=" * 70)
print("ALL-THREE-WRONG SUMMARY")
print("=" * 70)
print(f"  Evaluated (piece_id, task) pairs : {total_pairs}")
print(f"  Pairs ALL three models got wrong : {n_all_wrong}  ({n_all_wrong/total_pairs:.1%})")

print("\nBreakdown by task:")
for task, grp in all_wrong.groupby("question_type"):
    n_task = len(wide[has_all_models & (wide["question_type"] == task)])
    print(f"  {task:20s}: {len(grp)}/{n_task} wrong by all = {len(grp)/n_task:.1%}")

if "input_type" in all_wrong.columns:
    print("\nBreakdown by input type:")
    for itype, grp in all_wrong.groupby("input_type"):
        n_itype = len(wide[has_all_models & (wide["input_type"] == itype)])
        print(f"  {itype:6s}: {len(grp)}/{n_itype} = {len(grp)/n_itype:.1%}")

# ── PRINT DETAIL TABLE ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ALL-THREE-WRONG DETAIL TABLE")
print("=" * 70)

display_cols = group_keys + ["ground_truth"] + list(ANSWER_COLS.values())
print(all_wrong[display_cols].to_string(index=False))

# ── MOST COMMON WRONG ANSWERS (per task) ─────────────────────────────────────
print("\n" + "=" * 70)
print("MOST COMMON WRONG ANSWERS (across all models, for all-wrong cases)")
print("=" * 70)

answer_cols = list(ANSWER_COLS.values())

for task in all_wrong["question_type"].unique():
    sub = all_wrong[all_wrong["question_type"] == task]
    print(f"\n── {task} ──")
    print(f"  Ground truths involved: {sorted(sub['ground_truth'].unique().tolist())}")

    # Flatten all model answers into one series and count
    all_answers = pd.concat(
        [sub[col] for col in answer_cols if col in sub.columns]
    ).dropna().str.strip().str.lower()
    
    if all_answers.empty:
        print("  No parsed answers available.")
        continue

    counts = all_answers.value_counts().head(10)
    print("  Most frequent wrong answers:")
    for ans, cnt in counts.items():
        print(f"    '{ans}': {cnt} times")

# ── OPTIONAL: SAVE ────────────────────────────────────────────────────────────
if SAVE_CSV and n_all_wrong > 0:
    out_path = RESULTS_DIR / "all_three_wrong.csv"
    all_wrong[display_cols].to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")
elif n_all_wrong == 0:
    print("\nNo cases where all three models were wrong — great result!")

---
ALL-THREE-WRONG ANALYSIS — works across SRQ1, SRQ2, and SRQ3

HOW TO USE (paste each block into its own notebook cell):
- After SRQ1:
srq1_wrong = analyse_all_wrong_srq1()
- After SRQ2:
srq2_wrong = analyse_all_wrong_srq2()
- After SRQ3 (shows PDF vs PNG breakdown automatically):
srq3_wrong = analyse_all_wrong_srq3()
- All three in one go + cross-SRQ overlap:
run_all_wrong_analysis()


In [ ]:
# ── Model names must match what's in your JSONL files exactly ─────────────────
MODELS = ["claude-sonnet-4-6", "gemini-2.5-flash", "gpt-5.4"]

# ── Short labels for column names ─────────────────────────────────────────────
MODEL_SHORT = {
    "claude-sonnet-4-6": "claude",
    "gemini-2.5-flash":  "gemini",
    "gpt-5.4":           "gpt",
}


# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 1 — Shared helper functions
# ════════════════════════════════════════════════════════════════════════════════

def _load_jsonl(filepath):
    """Load a JSONL file into a DataFrame. Returns None if file is missing."""
    path = Path(filepath)
    if not path.exists():
        print(f"[ERROR] File not found: {filepath}")
        return None
    records = [json.loads(line) for line in path.open() if line.strip()]
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} entries from {path.name}")
    return df


def _drop_errors(df):
    """Remove rows where the model returned an API error."""
    n_before = len(df)
    df_clean = df[df["model_response"] != "ERROR"].copy()
    n_dropped = n_before - len(df_clean)
    if n_dropped:
        print(f"Dropped {n_dropped} ERROR entries ({n_dropped/n_before:.1%}) "
              f"→ {len(df_clean)} usable rows")
    return df_clean


def _check_models(df, label=""):
    """Warn if any expected model is missing from the loaded file."""
    found = df["model"].unique().tolist()
    missing = [m for m in MODELS if m not in found]
    if missing:
        print(f"[WARNING]{' ' + label if label else ''} "
              f"Missing models: {missing}. Found: {found}")
    return found


def _build_wide_table(df_clean, group_keys):
    """
    Pivot long-format results into one wide row per (piece_id, question_type
    [, input_type]), with separate columns for each model's correctness and
    parsed answer. Also attaches ground_truth.

    Parameters
    ----------
    df_clean   : DataFrame with ERROR rows already removed
    group_keys : list of column names that identify a unique item
                 e.g. ["piece_id", "question_type"] for SRQ1/2,
                      ["piece_id", "question_type", "input_type"] for SRQ3

    Returns
    -------
    wide : DataFrame — one row per item, columns:
           group_keys + ground_truth
           + <model>_correct  for each model
           + <model>_answer   for each model
    """
    # Keep only models that are actually present
    present_models = df_clean["model"].unique().tolist()

    # Pivot correctness
    pivot_correct = (
        df_clean
        .groupby(group_keys + ["model"])["correct"]
        .first()
        .unstack("model")
        .rename(columns={m: f"{MODEL_SHORT.get(m, m)}_correct"
                         for m in present_models})
        .reset_index()
    )

    # Pivot parsed answers
    pivot_answers = (
        df_clean
        .groupby(group_keys + ["model"])["parsed_answer"]
        .first()
        .unstack("model")
        .rename(columns={m: f"{MODEL_SHORT.get(m, m)}_answer"
                         for m in present_models})
        .reset_index()
    )

    # Ground truth (same value across models — take the first occurrence)
    gt_df = (
        df_clean
        .groupby(group_keys)["ground_truth"]
        .first()
        .reset_index()
    )

    wide = pivot_correct.merge(pivot_answers, on=group_keys)
    wide = wide.merge(gt_df, on=group_keys)
    return wide


def _find_all_wrong(wide, group_keys):
    """
    Given the wide table, return only rows where every model has
    correct == False (NaN rows are excluded — that means a model didn't run,
    not that it was wrong).

    Returns
    -------
    all_wrong    : filtered DataFrame
    total_pairs  : int — total rows with complete data for all models
    correct_cols : list of str — the correctness column names used
    answer_cols  : list of str — the parsed answer column names
    """
    correct_cols = [c for c in wide.columns if c.endswith("_correct")]
    answer_cols  = [c for c in wide.columns if c.endswith("_answer")]

    has_all = wide[correct_cols].notna().all(axis=1)
    all_wrong_mask = has_all & (wide[correct_cols] == False).all(axis=1)

    all_wrong   = wide[all_wrong_mask].copy().reset_index(drop=True)
    total_pairs = has_all.sum()
    return all_wrong, total_pairs, correct_cols, answer_cols


def _print_summary(all_wrong, total_pairs, group_keys, label, answer_cols):
    """Print headline numbers, per-task breakdown, and the detail table."""
    n = len(all_wrong)
    print("\n" + "=" * 70)
    print(f"{label} — ALL-THREE-WRONG SUMMARY")
    print("=" * 70)
    print(f"  Complete (piece_id, task) pairs : {total_pairs}")
    print(f"  All three models wrong          : {n}  "
          f"({n/total_pairs:.1%} of pairs)" if total_pairs else "  No complete pairs found.")

    if n == 0:
        print("\n  ✓ No cases where all three models were wrong.")
        return

    # Per-task breakdown
    print("\nBreakdown by question_type:")
    for task, grp in all_wrong.groupby("question_type"):
        # Denominator: complete pairs for this task
        n_task = total_pairs  # fallback
        print(f"  {task:22s}: {len(grp)} cases")

    # Per-input_type breakdown (SRQ3 only)
    if "input_type" in all_wrong.columns:
        print("\nBreakdown by input_type:")
        for itype, grp in all_wrong.groupby("input_type"):
            print(f"  {itype:6s}: {len(grp)} cases")

    # Detail table
    display_cols = group_keys + ["ground_truth"] + answer_cols
    display_cols = [c for c in display_cols if c in all_wrong.columns]
    print("\n" + "-" * 70)
    print("DETAIL TABLE")
    print("-" * 70)
    print(all_wrong[display_cols].to_string(index=False))

    # Most common wrong answers per task
    print("\n" + "-" * 70)
    print("MOST COMMON WRONG ANSWERS (per task, all models pooled)")
    print("-" * 70)
    for task in all_wrong["question_type"].unique():
        sub = all_wrong[all_wrong["question_type"] == task]
        print(f"\n  ── {task} ──")
        gt_vals = sorted(sub["ground_truth"].dropna().unique().tolist())
        print(f"  Ground truths involved : {gt_vals}")
        pooled = pd.concat(
            [sub[col] for col in answer_cols if col in sub.columns]
        ).dropna().str.strip().str.lower()
        if pooled.empty:
            print("  (no parsed answers)")
            continue
        for ans, cnt in pooled.value_counts().head(8).items():
            print(f"    '{ans}': {cnt}×")


def _save_csv(all_wrong, group_keys, answer_cols, out_path):
    """Save the all-wrong detail table to CSV."""
    if len(all_wrong) == 0:
        print("  Nothing to save (no all-wrong cases).")
        return
    display_cols = [c for c in (group_keys + ["ground_truth"] + answer_cols)
                    if c in all_wrong.columns]
    all_wrong[display_cols].to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")


# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 2 — SRQ-specific analysis functions
# ════════════════════════════════════════════════════════════════════════════════

def analyse_all_wrong_srq1(results_file=None, save_csv=True):
    """
    Find (piece_id, question_type) pairs where all 3 models were wrong.
    SRQ1 schema: zero-shot, PDF input only.

    Parameters
    ----------
    results_file : Path or str — defaults to SRQ1_RESULTS_FILE
    save_csv     : bool — write results/srq1_all_three_wrong.csv

    Returns
    -------
    DataFrame of all-wrong cases (empty DataFrame if none).
    """
    filepath = results_file or SRQ1_RESULTS_FILE
    group_keys = ["piece_id", "question_type"]

    df = _load_jsonl(filepath)
    if df is None:
        return pd.DataFrame()

    _check_models(df, label="SRQ1")
    df_clean = _drop_errors(df)

    wide = _build_wide_table(df_clean, group_keys)
    all_wrong, total_pairs, correct_cols, answer_cols = _find_all_wrong(wide, group_keys)

    _print_summary(all_wrong, total_pairs, group_keys, "SRQ1 (zero-shot, PDF)", answer_cols)

    if save_csv:
        _save_csv(all_wrong, group_keys, answer_cols,
                  RESULTS_DIR / "srq1_all_three_wrong.csv")

    return all_wrong


def analyse_all_wrong_srq2(results_file=None, save_csv=True):
    """
    Find (piece_id, question_type) pairs where all 3 models were wrong.
    SRQ2 schema: CoT prompting, PDF input — also shows reasoning snippets
    so you can see *why* all three failed.

    Parameters
    ----------
    results_file : Path or str — defaults to SRQ2_RESULTS_FILE
    save_csv     : bool — write results/srq2_all_three_wrong.csv

    Returns
    -------
    DataFrame of all-wrong cases (empty DataFrame if none).
    """
    filepath = results_file or SRQ2_RESULTS_FILE
    group_keys = ["piece_id", "question_type"]

    df = _load_jsonl(filepath)
    if df is None:
        return pd.DataFrame()

    _check_models(df, label="SRQ2")
    df_clean = _drop_errors(df)

    wide = _build_wide_table(df_clean, group_keys)
    all_wrong, total_pairs, correct_cols, answer_cols = _find_all_wrong(wide, group_keys)

    _print_summary(all_wrong, total_pairs, group_keys, "SRQ2 (CoT, PDF)", answer_cols)

    # ── SRQ2 EXTRA: show reasoning snippets for wrong cases ───────────────────
    # This helps you eyeball whether the reasoning was plausible but the final
    # answer was still wrong (category A in your coding scheme).
    if len(all_wrong) > 0 and "reasoning" in df_clean.columns:
        print("\n" + "-" * 70)
        print("REASONING SNIPPETS (first 200 chars per model, for all-wrong cases)")
        print("(Useful for manual coding: correct reasoning → wrong answer = Category A)")
        print("-" * 70)

        for _, row in all_wrong.iterrows():
            pid   = row["piece_id"]
            task  = row["question_type"]
            gt    = row["ground_truth"]
            print(f"\n  {pid} | {task} | ground truth: {gt}")

            for model in MODELS:
                short = MODEL_SHORT.get(model, model)
                # Pull reasoning from the original long-format df
                mask = (
                    (df_clean["piece_id"]       == pid) &
                    (df_clean["question_type"]   == task) &
                    (df_clean["model"]           == model)
                )
                rows = df_clean[mask]
                if rows.empty:
                    continue
                reasoning = rows.iloc[0].get("reasoning", "")
                snippet   = str(reasoning)[:200].replace("\n", " ") if reasoning else "(none)"
                ans       = row.get(f"{short}_answer", "—")
                print(f"    [{short}] answer='{ans}' | reasoning: {snippet}...")

    if save_csv:
        _save_csv(all_wrong, group_keys, answer_cols,
                  RESULTS_DIR / "srq2_all_three_wrong.csv")

    return all_wrong


def analyse_all_wrong_srq3(results_file=None, save_csv=True):
    """
    Find (piece_id, question_type, input_type) triples where all 3 models
    were wrong. SRQ3 schema: zero-shot + role prompt, PDF and PNG input.

    Also prints a cross-format comparison: cases wrong in BOTH formats vs
    only one format — useful for your input representation analysis.

    Parameters
    ----------
    results_file : Path or str — defaults to SRQ3_RESULTS_FILE
    save_csv     : bool — write results/srq3_all_three_wrong.csv

    Returns
    -------
    DataFrame of all-wrong cases (empty DataFrame if none).
    """
    filepath = results_file or SRQ3_RESULTS_FILE
    # SRQ3 has input_type as an extra grouping dimension
    group_keys = ["piece_id", "question_type", "input_type"]

    df = _load_jsonl(filepath)
    if df is None:
        return pd.DataFrame()

    _check_models(df, label="SRQ3")
    df_clean = _drop_errors(df)

    wide = _build_wide_table(df_clean, group_keys)
    all_wrong, total_pairs, correct_cols, answer_cols = _find_all_wrong(wide, group_keys)

    _print_summary(all_wrong, total_pairs, group_keys,
                   "SRQ3 (zero-shot + role, PDF + PNG)", answer_cols)

    # ── SRQ3 EXTRA: cross-format overlap ─────────────────────────────────────
    # For each (piece_id, question_type), was it wrong in both formats,
    # only PDF, or only PNG? This is the key SRQ3 question.
    if len(all_wrong) > 0 and "input_type" in all_wrong.columns:
        print("\n" + "-" * 70)
        print("CROSS-FORMAT OVERLAP")
        print("(For each item: was it all-wrong in PDF, PNG, or both?)")
        print("-" * 70)

        base_keys = ["piece_id", "question_type"]
        pdf_wrong = set(
            zip(
                all_wrong[all_wrong["input_type"] == "pdf"]["piece_id"],
                all_wrong[all_wrong["input_type"] == "pdf"]["question_type"]
            )
        )
        png_wrong = set(
            zip(
                all_wrong[all_wrong["input_type"] == "png"]["piece_id"],
                all_wrong[all_wrong["input_type"] == "png"]["question_type"]
            )
        )
        both_wrong    = pdf_wrong & png_wrong
        only_pdf_wrong = pdf_wrong - png_wrong
        only_png_wrong = png_wrong - pdf_wrong

        print(f"  Wrong in BOTH formats : {len(both_wrong)} items")
        print(f"  Wrong in PDF only     : {len(only_pdf_wrong)} items")
        print(f"  Wrong in PNG only     : {len(only_png_wrong)} items")

        if both_wrong:
            print("\n  Items all-wrong in BOTH PDF and PNG:")
            for pid, task in sorted(both_wrong):
                print(f"    {pid} | {task}")

        if only_pdf_wrong:
            print("\n  Items all-wrong in PDF only (PNG was fine for ≥1 model):")
            for pid, task in sorted(only_pdf_wrong):
                print(f"    {pid} | {task}")

        if only_png_wrong:
            print("\n  Items all-wrong in PNG only (PDF was fine for ≥1 model):")
            for pid, task in sorted(only_png_wrong):
                print(f"    {pid} | {task}")

    if save_csv:
        _save_csv(all_wrong, group_keys, answer_cols,
                  RESULTS_DIR / "srq3_all_three_wrong.csv")

    return all_wrong


# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 3 — Run all three + cross-SRQ overlap
# ════════════════════════════════════════════════════════════════════════════════

def run_all_wrong_analysis(save_csv=True):
    """
    Run all three SRQ all-wrong analyses in sequence, then print a cross-SRQ
    overlap table showing which (piece_id, question_type) pairs were hard
    across prompting strategies too.

    Returns
    -------
    dict with keys "srq1", "srq2", "srq3" → each a DataFrame of all-wrong cases.
    """
    print("Running all-wrong analysis across all SRQs...\n")

    srq1_wrong = analyse_all_wrong_srq1(save_csv=save_csv)
    print()
    srq2_wrong = analyse_all_wrong_srq2(save_csv=save_csv)
    print()
    srq3_wrong = analyse_all_wrong_srq3(save_csv=save_csv)

    # ── Cross-SRQ overlap ─────────────────────────────────────────────────────
    # Which (piece_id, question_type) pairs show up as all-wrong in SRQ1 AND SRQ2?
    # These are the genuinely hard items that neither zero-shot nor CoT could crack.
    print("\n" + "=" * 70)
    print("CROSS-SRQ OVERLAP — items all-wrong in multiple SRQs")
    print("=" * 70)

    def _to_set(df, keys=("piece_id", "question_type")):
        if df.empty:
            return set()
        return set(zip(*[df[k] for k in keys if k in df.columns]))

    s1 = _to_set(srq1_wrong)
    s2 = _to_set(srq2_wrong)
    # SRQ3 has input_type — collapse to (piece_id, question_type) for comparison
    s3 = _to_set(srq3_wrong)

    srq1_and_srq2 = s1 & s2
    srq1_and_srq3 = s1 & s3
    srq2_and_srq3 = s2 & s3
    all_three_srqs = s1 & s2 & s3

    print(f"\n  All-wrong in SRQ1 AND SRQ2 (zero-shot + CoT both failed): "
          f"{len(srq1_and_srq2)} items")
    for pid, task in sorted(srq1_and_srq2):
        print(f"    {pid} | {task}")

    print(f"\n  All-wrong in SRQ1 AND SRQ3 (zero-shot PDF + format comparison): "
          f"{len(srq1_and_srq3)} items")
    for pid, task in sorted(srq1_and_srq3):
        print(f"    {pid} | {task}")

    print(f"\n  All-wrong in SRQ2 AND SRQ3 (CoT + format comparison): "
          f"{len(srq2_and_srq3)} items")
    for pid, task in sorted(srq2_and_srq3):
        print(f"    {pid} | {task}")

    print(f"\n  All-wrong across ALL THREE SRQs (hardest items overall): "
          f"{len(all_three_srqs)} items")
    if all_three_srqs:
        for pid, task in sorted(all_three_srqs):
            print(f"    ★ {pid} | {task}")
    else:
        print("    (none — every item was solvable by at least one SRQ setup)")

    return {"srq1": srq1_wrong, "srq2": srq2_wrong, "srq3": srq3_wrong}

In [ ]:
srq1_wrong = analyse_all_wrong_srq1()   # zero-shot, PDF only
srq2_wrong = analyse_all_wrong_srq2()   # CoT, PDF — also prints reasoning snippets
srq3_wrong = analyse_all_wrong_srq3()   # PDF vs PNG — also prints cross-format overlap